# RAG Pipeline with Conversational Memory

## Overview
This notebook contains the full implementation of a Retrieval-Augmented Generation (RAG) pipeline with conversational memory, including testing and evaluation across multiple model configurations.

## Contents
- Full RAG pipeline implementation
- Predefined conversation test cell
- Interactive conversation test cell
- Evaluation across 3 embedding models and 1 generation model
  - Embedding models: MiniLM-L6-v2, BGE-base-en-v1.5, Multi-QA-MPNet-base
  - Generation model: Qwen2.5-7B-Instruct

### 1. Full RAG pipeline implementation

In [2]:
import json
import re
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from typing import List, Dict, Any, Set, Tuple, Optional
from collections import defaultdict
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import CrossEncoder
import os
import logging
import warnings

os.environ["TRANSFORMERS_VERBOSITY"] = "error"
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)


from dotenv import load_dotenv
load_dotenv()
token = os.environ.get("HF_TOKEN")

# ============================================================================
# RERANKER CLASS
# ============================================================================
class Reranker:
    def __init__(self, use_cross_encoder=True, cross_encoder_model='cross-encoder/ms-marco-MiniLM-L-6-v2'):
        """
        Initialize the reranker with multiple scoring strategies
        
        Args:
            use_cross_encoder: Whether to use a cross-encoder model for reranking
            cross_encoder_model: Name of the cross-encoder model to use
        """
        self.use_cross_encoder = use_cross_encoder
        
        # Initialize Cross-Encoder if enabled
        if use_cross_encoder:
            try:
                print(f"Loading cross-encoder model: {cross_encoder_model}")
                self.cross_encoder = CrossEncoder(cross_encoder_model)
                print("Cross-encoder loaded successfully!")
            except Exception as e:
                print(f"Failed to load cross-encoder: {e}")
                print("Falling back to TF-IDF based reranking")
                self.use_cross_encoder = False
                self.cross_encoder = None
        else:
            self.cross_encoder = None
        
        # Initialize TF-IDF vectorizer for fallback/hybrid scoring
        self.tfidf_vectorizer = TfidfVectorizer(
            max_features=1000,
            stop_words='english',
            ngram_range=(1, 2)
        )
        self.tfidf_matrix = None
    
    def compute_keyword_score(self, query: str, chunk: str) -> float:
        """Compute keyword-based relevance score"""
        query_lower = query.lower()
        chunk_lower = chunk.lower()
        
        # Tokenize
        query_tokens = set(word_tokenize(query_lower))
        chunk_tokens = set(word_tokenize(chunk_lower))
        
        # Compute overlap
        overlap = len(query_tokens.intersection(chunk_tokens))
        
        # Normalize by query length
        if len(query_tokens) == 0:
            return 0.0
        
        keyword_score = overlap / len(query_tokens)
        
        # Boost for exact phrase matches
        if query_lower in chunk_lower:
            keyword_score += 0.5
        
        # Boost for Q&A format that contains the query
        if "Q:" in chunk and "A:" in chunk:
            question_part = chunk.split("A:")[0] if "A:" in chunk else chunk
            if any(token in question_part.lower() for token in query_tokens):
                keyword_score += 0.3
        
        return min(keyword_score, 1.0)
    
    def compute_tfidf_score(self, query: str, chunks: List[str]) -> List[float]:
        """Compute TF-IDF based similarity scores"""
        try:
            # Combine query with chunks for vectorization
            all_texts = [query] + chunks
            
            # Fit and transform
            tfidf_matrix = self.tfidf_vectorizer.fit_transform(all_texts)
            
            # Compute cosine similarity between query and each chunk
            query_vector = tfidf_matrix[0:1]
            chunk_vectors = tfidf_matrix[1:]
            
            similarities = cosine_similarity(query_vector, chunk_vectors)[0]
            
            return similarities.tolist()
        except:
            # Return zeros if TF-IDF fails
            return [0.0] * len(chunks)
    
    def compute_position_score(self, position: int, total: int) -> float:
        """Compute position-based score (earlier positions get higher scores)"""
        if total == 0:
            return 0.0
        # Exponential decay based on position
        return np.exp(-2.0 * position / total)
    
    def compute_qa_format_score(self, query: str, chunk: str) -> float:
        """Score based on Q&A format alignment"""
        score = 0.0
        query_lower = query.lower()
        
        # Check if chunk is in Q&A format
        if "Q:" in chunk and "A:" in chunk:
            score += 0.2
            
            # Extract question part
            question = chunk.split("A:")[0].replace("Q:", "").strip().lower()
            
            # Check for similar question patterns
            query_words = set(word_tokenize(query_lower))
            question_words = set(word_tokenize(question))
            
            overlap = len(query_words.intersection(question_words))
            if len(query_words) > 0:
                similarity = overlap / len(query_words)
                score += similarity * 0.3
        
        # Check for program-specific patterns
        if any(pattern in query_lower for pattern in ['tuition', 'fee', 'cost']):
            if 'tuition' in chunk.lower() or 'fee' in chunk.lower():
                score += 0.2
        
        if any(pattern in query_lower for pattern in ['deadline', 'application period']):
            if 'deadline' in chunk.lower() or 'application' in chunk.lower():
                score += 0.2
        
        if any(pattern in query_lower for pattern in ['requirement', 'prerequisite']):
            if 'requirement' in chunk.lower() or 'prerequisite' in chunk.lower():
                score += 0.2
        
        return score
    
    def rerank_chunks(self, query: str, chunks: List[str], top_k: int = None, verbose: bool = False) -> List[Tuple[str, float]]:
        """
        Rerank chunks based on multiple scoring strategies
        
        Returns:
            List of (chunk, score) tuples sorted by score descending
        """
        if not chunks:
            return []
        
        chunk_scores = []
        
        # Get TF-IDF scores
        tfidf_scores = self.compute_tfidf_score(query, chunks)
        
        for i, chunk in enumerate(chunks):
            score_components = {}
            
            # 1. Cross-encoder score (if available)
            if self.use_cross_encoder and self.cross_encoder:
                try:
                    cross_encoder_score = self.cross_encoder.predict([[query, chunk]])[0]
                    # Normalize cross-encoder score to [0, 1]
                    score_components['cross_encoder'] = (cross_encoder_score + 1) / 2
                except:
                    score_components['cross_encoder'] = 0.0
            else:
                score_components['cross_encoder'] = 0.0
            
            # 2. Keyword score
            score_components['keyword'] = self.compute_keyword_score(query, chunk)
            
            # 3. TF-IDF score
            score_components['tfidf'] = tfidf_scores[i]
            
            # 4. Position score
            score_components['position'] = self.compute_position_score(i, len(chunks))
            
            # 5. Q&A format score
            score_components['qa_format'] = self.compute_qa_format_score(query, chunk)
            
            # Combine scores with weights
            if self.use_cross_encoder and self.cross_encoder:
                # With cross-encoder: rely heavily on it
                final_score = (
                    score_components['cross_encoder'] * 0.5 +
                    score_components['keyword'] * 0.2 +
                    score_components['tfidf'] * 0.1 +
                    score_components['position'] * 0.1 +
                    score_components['qa_format'] * 0.1
                )
            else:
                # Without cross-encoder: use other signals
                final_score = (
                    score_components['keyword'] * 0.35 +
                    score_components['tfidf'] * 0.25 +
                    score_components['position'] * 0.15 +
                    score_components['qa_format'] * 0.25
                )
            
            chunk_scores.append((chunk, final_score, score_components))
        
        # Sort by final score descending
        chunk_scores.sort(key=lambda x: x[1], reverse=True)
        
        # Return top_k if specified
        if top_k:
            chunk_scores = chunk_scores[:top_k]
        
        # Return (chunk, score) tuples
        return [(chunk, score) for chunk, score, _ in chunk_scores]


# ============================================================================
# INTELLIGENT CONVERSATION MEMORY CLASS
# ============================================================================
class ConversationMemory:
    """
    Simple conversation memory - stores ALL turns in session
    Passes ENTIRE conversation history to LLM for natural understanding
    NO JSON parsing - just natural language!
    """
    
    def __init__(self, embedding_model, llm_model, llm_tokenizer):
        self.embedding_model = embedding_model
        self.llm_model = llm_model
        self.llm_tokenizer = llm_tokenizer
        
        # Full conversation storage - ALL turns in session
        self.conversation_turns = []
        
        print("Conversation Memory initialized (Session-wise - ALL turns)!")
    
    def add_turn(self, query: str, answer: str, metadata: dict = None):
        """Store conversation turn"""
        turn_id = len(self.conversation_turns)
        
        turn_data = {
            'turn_id': turn_id,
            'query': query,
            'answer': answer,
            'timestamp': time.time(),
            'metadata': metadata or {}
        }
        
        self.conversation_turns.append(turn_data)
    
    def get_all_turns(self) -> List[dict]:
        """Get ALL conversation turns from this session"""
        return self.conversation_turns
    
    def get_conversation_summary(self) -> str:
        """Get a summary of the conversation"""
        if not self.conversation_turns:
            return "No conversation history"
        
        total_turns = len(self.conversation_turns)
        recent_turn = self.conversation_turns[-1]
        
        return f"Total turns: {total_turns} | Last query: {recent_turn['query'][:50]}..."
    
    def clear_session(self):
        """Clear all conversation history (start new session)"""
        self.conversation_turns = []
        print("Session cleared")


# ============================================================================
# BACKGROUND MATCHER CLASS
# ============================================================================

class LLMQueryRouter:
    """
    Replaces DynamicBackgroundMatcher.
    Uses a single LLM call to classify intent and extract parameters,
    then routes to the appropriate handler.
    """

    # Maps LLM output keys → actual department keys in course_data
    # Add or adjust these to match exactly what's in your course_data department field
    DEPT_KEY_MAP = {
        "cs":              ["Computer Science"],
        "business":        ["Business"],
        "engineering":     ["Engineering"],
        "graduate_school": ["Graduate School"],
    }

    # These departments also show Graduate School masters
    GRADUATE_SCHOOL_DEPTS = {"business", "engineering"}

    def __init__(self, course_data: dict, model, tokenizer):
        self.course_data = course_data
        self.model = model
        self.tokenizer = tokenizer
        self.program_index = self._build_program_index()

        # Print what department keys actually exist in the data (debug once at startup)
        actual_depts = set()
        for details in course_data.values():
            dept = details.get("department", "").strip()
            if dept:
                actual_depts.add(dept.lower())
        

    # ------------------------------------------------------------------
    # Index builder — stores programs under the EXACT dept string from data
    # ------------------------------------------------------------------

    def _build_program_index(self) -> dict:
        from collections import defaultdict

        index = {
            "by_department": defaultdict(lambda: {"bachelor": [], "master": []}),
            "all_programs":  {"bachelor": [], "master": []},
            "by_language":   defaultdict(lambda: {"bachelor": [], "master": []}),
            "by_tuition":    defaultdict(lambda: {"bachelor": [], "master": []}),
        }

        for program_name, details in self.course_data.items():
            program_type = details.get("program_type", "").lower()
            level = "master" if "master" in program_type else "bachelor"

            index["all_programs"][level].append(program_name)

            # Store under exact dept string (preserving case to match course_data)
            dept = details.get("department", "").strip()
            if dept:
                index["by_department"][dept][level].append(program_name)

            lang = details.get("language_of_instruction", "").lower()
            if "english" in lang:
                index["by_language"]["english"][level].append(program_name)
            elif "german" in lang:
                index["by_language"]["german"][level].append(program_name)

            tuition = details.get("tuition_fees", "").lower()
            is_free = not tuition or "none" in tuition or "free" in tuition or "no tuition" in tuition
            key = "free" if is_free else "paid"
            index["by_tuition"][key][level].append(program_name)

        return index

    # ------------------------------------------------------------------
    # Resolve LLM dept key → actual dept strings in the index
    # ------------------------------------------------------------------

    def _resolve_dept_keys(self, llm_dept_key: str) -> list:
        """
        Converts LLM output like 'cs' into the actual department strings
        used in the index, e.g. ['computer science', 'informatics'].
        Falls back to direct lookup if key not in map.
        """
        return self.DEPT_KEY_MAP.get(llm_dept_key, [llm_dept_key])

    # ------------------------------------------------------------------
    # Single LLM call: classify intent + extract parameters
    # ------------------------------------------------------------------

    def _classify(self, query: str) -> dict:
        prompt = f"""<|im_start|>system
        You are a query classifier for a university chatbot. Analyze the user query and output JSON only.
        
        INTENTS:
        - FILTER: user wants to list/filter programs (by level, language, tuition, department). No specific program named.
        - DISCOVER: user wants to explore what programs exist based on their background, field of interest, or degree. No specific program named.
        - ELIGIBILITY: user asks if they can apply to or are qualified for a SPECIFIC named program.
        - OUT_OF_SCOPE: question is completely unrelated to Hof University (sports, weather, news, cooking, general knowledge, etc.)
        - OTHER: anything else related to Hof University (exam rules, deadlines, fees, contact info, etc.)
        
        DEPARTMENTS (use these exact keys):
        - cs             → computer science, informatics, software, AI, robotics, data science
        - business       → business, management, economics, finance, marketing, MBA
        - engineering    → engineering, mechanical, electrical, civil, industrial, textiles
        - graduate_school → graduate school, postgraduate combined business+engineering masters
        
        RULES:
        - If a specific program name is mentioned AND the user asks about their qualifications → ELIGIBILITY
        - If a specific program name is mentioned but NO qualifications → OTHER
        - If user mentions their background/degree/field but NO specific program → DISCOVER
        - If user asks to list/show/filter programs → FILTER
        - FILTER requires explicit listing intent like "show me", "list", "what programs are available". Do NOT use FILTER for questions about a specific program's details.
        - Questions about fees, deadlines, language, requirements, duration, contact for "this program" or "it" → OTHER
        - graduate_school is ONLY for queries explicitly about the Graduate School itself
        
        Output ONLY valid JSON, no explanation:
        {{
          "intent": "...",
          "level": null,
          "language": null,
          "tuition": null,
          "department": null,
          "program": null
        }}<|im_end|>
        <|im_start|>user
        {query}<|im_end|>
        <|im_start|>assistant
        """
        try:
            inputs = self.tokenizer(
                prompt, return_tensors="pt", truncation=True, max_length=1024
            )
            device = next(self.model.parameters()).device
            inputs = {k: v.to(device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=120,
                    temperature=0.1,
                    do_sample=True,
                    pad_token_id=self.tokenizer.pad_token_id or self.tokenizer.eos_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )

            input_length = inputs["input_ids"].shape[1]
            generated = self.tokenizer.decode(
                outputs[0][input_length:], skip_special_tokens=True
            )
            generated = generated.replace("<|im_end|>", "").strip()

            start = generated.find("{")
            end   = generated.rfind("}") + 1
            if start != -1 and end > start:
                result = json.loads(generated[start:end])
                return result

        except Exception as e:
            print(f"[ROUTER CLASSIFY ERROR] {e}")

        return {"intent": "OTHER", "level": None, "language": None,
                "tuition": None, "department": None, "program": None}

    # ------------------------------------------------------------------
    # Main entry point
    # ------------------------------------------------------------------

    def route(self, query: str):
        params = self._classify(query)
        intent = params.get("intent", "OTHER")

        if intent == "FILTER":
            return intent, self._handle_filter(query, params)

        if intent == "DISCOVER":
            return intent, self._handle_discover(query, params)

        return intent, None

    # ------------------------------------------------------------------
    # FILTER handler
    # ------------------------------------------------------------------

    def _handle_filter(self, query: str, params: dict) -> str:
        level      = params.get("level")
        language   = params.get("language")
        tuition    = params.get("tuition")
        llm_dept   = params.get("department")

        if level:
            programs = set(self.program_index["all_programs"][level])
        else:
            programs = set(
                self.program_index["all_programs"]["bachelor"] +
                self.program_index["all_programs"]["master"]
            )

        if language:
            lang_pool = set(
                self.program_index["by_language"][language]["bachelor"] +
                self.program_index["by_language"][language]["master"]
            )
            programs &= lang_pool

        if tuition == "free":
            free_pool = set(
                self.program_index["by_tuition"]["free"]["bachelor"] +
                self.program_index["by_tuition"]["free"]["master"]
            )
            programs &= free_pool

        if llm_dept:
            dept_pool = set()
            for actual_dept in self._resolve_dept_keys(llm_dept):
                entry = self.program_index["by_department"].get(actual_dept, {})
                dept_pool.update(entry.get("bachelor", []))
                dept_pool.update(entry.get("master", []))
            programs &= dept_pool

        if not programs:
            return self._llm_format(query, "No programs found matching the specified filters.", programs=[])

        bachelors = sorted(p for p in programs if p in self.program_index["all_programs"]["bachelor"])
        masters   = sorted(p for p in programs if p in self.program_index["all_programs"]["master"])

        return self._llm_format(query, None, programs={"bachelor": bachelors, "master": masters})

    # ------------------------------------------------------------------
    # DISCOVER handler
    # ------------------------------------------------------------------

    def _handle_discover(self, query: str, params: dict) -> str:
        llm_dept = params.get("department")
        level    = params.get("level")

        if not llm_dept:
            return self._llm_ask_department(query)

        # Collect all actual dept keys to query
        target_actual_depts = set(self._resolve_dept_keys(llm_dept))

        # Also include graduate_school if dept is business or engineering
        if llm_dept in self.GRADUATE_SCHOOL_DEPTS:
            target_actual_depts.update(self._resolve_dept_keys("graduate_school"))

        programs_found = {"bachelor": [], "master": []}

        for actual_dept in target_actual_depts:
            dept_entry = self.program_index["by_department"].get(actual_dept, {})
            if not level or level == "bachelor":
                programs_found["bachelor"].extend(dept_entry.get("bachelor", []))
            if not level or level == "master":
                programs_found["master"].extend(dept_entry.get("master", []))

        programs_found["bachelor"] = sorted(set(programs_found["bachelor"]))
        programs_found["master"]   = sorted(set(programs_found["master"]))



        if not programs_found["bachelor"] and not programs_found["master"]:
            return self._llm_format(
                query,
                f"No programs found for the detected department: {llm_dept}. Please try asking about Computer Science, Business, or Engineering.",
                programs=[]
            )

        return self._llm_format(query, None, programs=programs_found)

    # ------------------------------------------------------------------
    # LLM formatting
    # ------------------------------------------------------------------

    def _llm_format(self, query: str, fallback_message, programs) -> str:
        if fallback_message:
            program_text = fallback_message
        elif isinstance(programs, dict):
            lines = []
            if programs.get("bachelor"):
                lines.append("Bachelor Programs:")
                lines.extend(f"  - {p}" for p in programs["bachelor"])
            if programs.get("master"):
                lines.append("Master Programs:")
                lines.extend(f"  - {p}" for p in programs["master"])
            program_text = "\n".join(lines) if lines else "No programs found."
        else:
            program_text = "No programs found."

        prompt = f"""<|im_start|>system
        You are a helpful assistant for Hof University. Present the program list naturally and helpfully.
        STRICT RULES:
        - List ONLY the programs provided. Do not add any programs.
        - Do NOT add any commentary about tuition fees, costs, or financial aid unless it is explicitly stated in the program list.
        - Do NOT say programs "come with fees" or "are not free" — you do not have that information.
        - Do NOT suggest checking scholarships or financial aid unless the user asked about it.
        - Keep the response short: one introductory sentence, then the numbered list, nothing after.
        - Start each numbered item at the beginning of the line with no indentation.
        - FORMATTING: Use numbered lists (1. 2. 3.) only. No asterisks, no bullet points, no bold text.<|im_end|>
        <|im_start|>user
        User asked: {query}
        
        Available programs:
        {program_text}
        
        Write a short, natural response listing these programs.<|im_end|>
        <|im_start|>assistant
        """
        return self._generate(prompt, max_new_tokens=400)

    # ------------------------------------------------------------------
    # LLM clarification
    # ------------------------------------------------------------------

    def _llm_ask_department(self, query: str) -> str:
        return (
            "I couldn't find an exact match for your background at Hof University. "
            "However, Hof University offers programs from the following 4 departments. "
            "Could you tell me if you have an interest in any of these areas?\n\n"
            "1. Business\n"
            "2. Engineering\n"
            "3. Computer Science\n"
            "4. Graduate School (combined Business and Engineering Master's programs)"
        )

    # ------------------------------------------------------------------
    # Shared generate helper
    # ------------------------------------------------------------------

    def _generate(self, prompt: str, max_new_tokens: int = 200) -> str:
        try:
            inputs = self.tokenizer(
                prompt, return_tensors="pt", truncation=True, max_length=2048
            )
            device = next(self.model.parameters()).device
            inputs = {k: v.to(device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    temperature=0.3,
                    top_p=0.85,
                    do_sample=True,
                    repetition_penalty=1.2,
                    pad_token_id=self.tokenizer.pad_token_id or self.tokenizer.eos_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )

            input_length = inputs["input_ids"].shape[1]
            result = self.tokenizer.decode(
                outputs[0][input_length:], skip_special_tokens=True
            )
            return result.replace("<|im_end|>", "").strip()

        except Exception as e:
            print(f"[ROUTER GENERATE ERROR] {e}")
            return "I'm sorry, I couldn't generate a response. Please try again."


            


# ============================================================================
# RAG PIPELINE 
# ============================================================================

class RAGPipeline:
    """RAG Pipeline with conversation memory"""
    
    def __init__(self, model, tokenizer, embedding_model, generation_model, 
                 faiss_index, all_chunks, course_data, background_matcher, 
                 enable_reranking=True, reranker_top_k=5, enable_memory=True):
        
        self.model = model
        self.tokenizer = tokenizer
        self.embedding_model = embedding_model
        self.faiss_index = faiss_index
        self.all_chunks = all_chunks
        self.course_data = course_data
        self.background_matcher = background_matcher
        self.use_llm = model is not None and tokenizer is not None
        
        # Reranking
        self.enable_reranking = enable_reranking
        self.reranker_top_k = reranker_top_k
        self.reranker = None
        
        if enable_reranking:
            print("Initializing Reranker...")
            self.reranker = Reranker(
                use_cross_encoder=True,
                cross_encoder_model='cross-encoder/ms-marco-MiniLM-L-6-v2'
            )
            print("Reranker initialized!")
        
        # Conversation Memory
        self.enable_memory = enable_memory and self.use_llm
        if self.enable_memory:
            self.conversation_memory = ConversationMemory(
                embedding_model=embedding_model,
                llm_model=model,
                llm_tokenizer=tokenizer
            )
        else:
            self.conversation_memory = None
            print("Conversation Memory disabled (requires LLM)")




    def answer_query(self, query: str, verbose: bool = False) -> str:
        """
        Main entry point - detects and handles multiple questions
        """
        
        # Detect if multiple questions
        questions = self.detect_multiple_questions(query)
        
        if len(questions) > 1:
            if verbose:
                print(f"\n[MULTI-QUESTION] Split into {len(questions)} questions")
                for i, q in enumerate(questions, 1):
                    print(f"  Q{i}: {q}")
            
            # Answer each question
            answers = []
            for i, q in enumerate(questions, 1):
                answer = self._answer_single_query(q, verbose=verbose)
                answers.append(f"Question {i}: {q}\n\n{answer}")
            
            # Combine all answers
            combined_answer = "\n\n---\n\n".join(answers)
            
            # Store in memory
            if self.enable_memory:
                self.conversation_memory.add_turn(
                    query=query,
                    answer=combined_answer,
                    metadata={'multi_question': True, 'count': len(questions)}
                )
            
            return combined_answer
        
        # Single question
        return self._answer_single_query(query, verbose=verbose)

    
    
    def _answer_single_query(self, query: str, verbose: bool = False) -> str:
        """
        Answer a single query. Uses LLMQueryRouter for intent detection,
        falls through to RAG for ELIGIBILITY and OTHER.
        """
        
        self._last_resolved_query = query  # default: no resolution = original query

        
    
        # ── Step 1: Route via LLM ──────────────────────────────────────────
        intent, routed_answer = self.background_matcher.route(query)
    
    
        # FILTER and DISCOVER are fully handled by the router
        if routed_answer is not None:
            if self.enable_memory:
                self.conversation_memory.add_turn(
                    query=query,
                    answer=routed_answer,
                    metadata={"type": intent.lower()}
                )
            return routed_answer
    
        # ── Step 2: RAG path (ELIGIBILITY + OTHER) ────────────────────────
        all_turns = []
        if self.enable_memory:
            all_turns = self.conversation_memory.get_all_turns()
            if verbose:
                print(f"[MEMORY] {len(all_turns)} previous turns")
    
        # Resolve pronouns/references before retrieval
        query_for_retrieval = query
        if all_turns and self.use_llm:
            try:
                query_for_retrieval = self.resolve_query_with_context(query, all_turns)
                if verbose and query_for_retrieval != query:
                    print(f"[RESOLVED] {query} → {query_for_retrieval}")
            except Exception as e:
                if verbose:
                    print(f"[RESOLVE ERROR] {e}")
    
        # Store resolved query so evaluator can access it
        self._last_resolved_query = query_for_retrieval
    
        # Retrieve
        context_chunks = self.retrieve_contexts(query_for_retrieval, k=500, verbose=verbose)
    
        if not context_chunks:
            answer = "I couldn't find relevant information. Please contact the admissions office."
            if self.enable_memory:
                self.conversation_memory.add_turn(query=query, answer=answer)
            return answer
    
        # Generate
        context = "\n\n".join(context_chunks)
        answer = self.generate_llm_answer_with_context(
            query=query,
            knowledge_context=context,
            memory_turns=all_turns,
            verbose=verbose
        )
    
        # Append a link if available
        import re
        links = re.findall(r"(https://[^\s]+)", context)
        if links and "http" not in answer:
            answer += f"\n\nFor more information: {links[0]}"
    
        if self.enable_memory:
            self.conversation_memory.add_turn(
                query=query, answer=answer, metadata={"type": "rag"}
            )
    
        return self.clean_answer(answer)



    def detect_multiple_questions(self, query: str) -> List[str]:
        """
        Use LLM to detect and split multiple questions
        """
        if not self.use_llm:
            return [query]
        
        # Quick heuristic: if no clear multi-question indicators, skip LLM
        if '?' not in query or (query.count('?') == 1 and ' and ' not in query.lower() and ' also ' not in query.lower()):
            return [query]
        
        # Get conversation context
        conversation_context = ""
        if self.enable_memory:
            all_turns = self.conversation_memory.get_all_turns()
            if all_turns:
                # Include first turn for context about what program they're asking about
                first_turn = all_turns[0]
                conversation_context = f"USER'S FIRST QUESTION: {first_turn['query']}\n\n"
        
        detection_prompt = f"""<|im_start|>system
        You analyze questions to detect if they contain multiple separate questions.<|im_end|>
        <|im_start|>user
        {conversation_context}CURRENT QUERY: "{query}"
        
        TASK: Determine if this contains multiple questions.
        
        RULES:
        1. If ONE question → output: SINGLE
        2. If MULTIPLE questions → list each on a new line starting with "Q:"
        3. When splitting, add context from the conversation so each question is complete
        4. Examples of multiple questions:
           - "What's the tuition fee? Also the deadline?"
           - "Can I apply for AI program and what's the cost?"
        
        OUTPUT:<|im_end|>
        <|im_start|>assistant
        """
        
        try:
            inputs = self.tokenizer(detection_prompt, return_tensors="pt", truncation=True, max_length=1024)
            device = next(self.model.parameters()).device
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=250,
                    temperature=0.1,
                    do_sample=True,
                    pad_token_id=self.tokenizer.pad_token_id if self.tokenizer.pad_token_id else self.tokenizer.eos_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )
            
            # Only decode new tokens
            input_length = inputs['input_ids'].shape[1]
            generated_tokens = outputs[0][input_length:]
            result = self.tokenizer.decode(generated_tokens, skip_special_tokens=True)
            
            result = result.replace("<|im_end|>", "").strip()
            
            # Parse result
            if "SINGLE" in result.upper()[:20]:
                return [query]
            
            # Extract questions starting with "Q:"
            questions = []
            for line in result.split('\n'):
                line = line.strip()
                if line.startswith('Q:'):
                    q = line[2:].strip().strip('"\'')
                    if q:
                        questions.append(q)
            
            return questions if len(questions) > 1 else [query]
            
        except Exception as e:
            print(f"[MULTI-Q ERROR] {e}")
            return [query]

        
    
    def retrieve_contexts(self, query: str, k: int = 20, 
                      top_k_override: int = None, verbose: bool = False) -> List[str]:
    
        query_embedding = self.embedding_model.encode([query])
        distances, indices = self.faiss_index.search(query_embedding, k)
        retrieved_chunks = [self.all_chunks[i] for i in indices[0]]
        
        # Deduplicate
        unique_chunks = []
        seen_content = set()
        for chunk in retrieved_chunks:
            normalized = re.sub(r'\s+', ' ', chunk.lower()).strip()
            if normalized not in seen_content:
                unique_chunks.append(chunk)
                seen_content.add(normalized)
        

        
        effective_top_k = top_k_override or self.reranker_top_k
        
        if self.enable_reranking and self.reranker:
            reranked_results = self.reranker.rerank_chunks(
                query=query,
                chunks=unique_chunks,
                top_k=effective_top_k,
                verbose=verbose
            )
            

            return [chunk for chunk, score in reranked_results]
        else:
            return unique_chunks[:12]

        
    
    def resolve_query_with_context(self, query: str, conversation_turns: List[dict]) -> str:
        """
        Resolve pronouns and references using conversation history
        """
        if not conversation_turns:
            return query
        
        # Build conversation history with emphasis on the FIRST question
        conversation_text = "CONVERSATION:\n\n"
        
        # Highlight the FIRST user question (most important for context)
        if len(conversation_turns) > 0:
            first_turn = conversation_turns[0]
            conversation_text += f"ORIGINAL USER QUESTION: {first_turn['query']}\n"
            answer_preview = first_turn['answer']
            conversation_text += f"Assistant: {answer_preview}\n\n"
        
        # Add recent turns
        if len(conversation_turns) > 1:
            conversation_text += "RECENT MESSAGES:\n"
            for turn in conversation_turns:
                conversation_text += f"User: {turn['query']}\n"
                answer_text = turn['answer']
                conversation_text += f"Assistant: {answer_text}\n\n"
        
        # Enhanced prompt that emphasizes original context
        resolve_prompt = f"""<|im_start|>system
        You are a query rewriting assistant. Your job is to make vague questions specific. BUT only when needed.<|im_end|>
        <|im_start|>user
        {conversation_text}
        
        CURRENT QUESTION: {query}
        
        RULES:
        1. Look at the USER'S ORIGINAL QUESTION to find what program/person they asked about
        2. Determine if this question needs context from the conversation.
        3. Rewrite incomplete questions:
           - "Also the deadline?" → "What is the application deadline for [program from original question]?"
           - "addmission requirements for the 1st program?"  → "What is the addmission requirements for [first item from the assistant's answer]?"
           - "Also the tuition?" → "What is the tuition fee for [program from original question]?"
           - "this program", "it" , "is this", "for this" → use the specific program from the ORIGINAL question
           - "his", "her" → use the person's name
           - "the 1st" → use the first item from the list
           - "Do I need health insurance to admit this program?" -  Here you do not need to resolved this to specific program. rather rewrite as "Is health insurance mandatory?"

        4. ALWAYS STRIP personal qualifications (CGPA, GPA, IELTS score, TOEFL, grades, background, experience).
           - "Can I apply with CGPA 1.7 and IELTS 7?" → "What are the admission requirements for [program]?"
           - "Am I eligible with a 3.5 GPA?" → "What are the admission requirements for [program]?"
           - "I have B2 German, can I apply?" → "What are the language requirements for [program]?"

        5. EXAMPLES OF QUESTIONS THAT DON'T NEED CONTEXT (already complete, general university questions):
           - "Can I work while studying?" → UNCHANGED
           - "What are the tuition fees at Hof University?" → UNCHANGED
           - "How do I apply for a student visa?" → UNCHANGED
           - "Are there any scholarships available?" → UNCHANGED
           - "What scholarships are available?" → UNCHANGED
           - "What are the basic admission requirements for bachelor programs?" → UNCHANGED
           - "What are the basic admission requirements for master programs?" → UNCHANGED
           - "What are the general admission requirements?" → UNCHANGED
           - Any question containing "bachelor programs" or "master programs" (plural) → UNCHANGED
           - If the current question is already complete and self-contained (contains a specific subject, program name, or department)→ UNCHANGED
  
        
        CRITICAL: 
        - Questions starting with "Also" are INCOMPLETE and MUST be expanded
        - Use the program mentioned in the USER'S ORIGINAL QUESTION
        - Don't pick random programs from the assistant's answer
        
        OUTPUT: REWRITTEN QUESTION (or UNCHANGED if already complete) <|im_end|>
        <|im_start|>assistant
        """
        
        try:
            inputs = self.tokenizer(resolve_prompt, return_tensors="pt", truncation=True, max_length=1536)
            device = next(self.model.parameters()).device
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=100,
                    temperature=0.1,  # Low temperature for consistency
                    do_sample=True,
                    pad_token_id=self.tokenizer.pad_token_id if self.tokenizer.pad_token_id else self.tokenizer.eos_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )
            
            # Only decode the NEW tokens
            input_length = inputs['input_ids'].shape[1]
            generated_tokens = outputs[0][input_length:]
            resolved = self.tokenizer.decode(generated_tokens, skip_special_tokens=True)
            
            # Clean
            resolved = resolved.replace("<|im_end|>", "").strip()
            resolved = resolved.strip('"\'.:,')
            resolved = resolved.replace("OUTPUT:", "").strip()
            
            # Validation
            if len(resolved) < 5 or len(resolved) > len(query) * 3:
                return query
            
            return resolved
            
        except Exception as e:
            print(f"[RESOLVE ERROR] {e}")
            return query

        
    
    def generate_llm_answer_with_context(self, query: str, knowledge_context: str,
                                    memory_turns: List[dict] = None, verbose: bool = False) -> str:
        """
        Generate answer using resolved query and knowledge base
        """
        
        if not self.use_llm:
            return self.extract_fallback_answer(knowledge_context)
        
        resolved_query = query
        
        # Resolve references if we have conversation history
        if memory_turns and len(memory_turns) > 0:
            resolved_query = self.resolve_query_with_context(query, memory_turns)
            

        
        # Qwen chat format
        answer_prompt = f"""<|im_start|>system
        You are a helpful assistant for Hof University. Answer questions accurately using only the provided information. If the question has nothing to do with Hof University, studying, or academic life, any kind of news, sports → answer something like 
        "I'm only able to answer questions related to Hof University, such as programs, admissions, fees, and academic regulations. For other topics, please refer to the appropriate resources.<|im_end|>
        <|im_start|>user
        Context: {knowledge_context}
        
        Question: {resolved_query}
         Start your answer directly with the relevant information. If the context only has partial information, share what you know and suggest contacting the admissions office for the rest. Never start with "I don't have that information."<|im_end|>

        <|im_start|>assistant
        """
        
        try:
            inputs = self.tokenizer(answer_prompt, return_tensors="pt", truncation=True, max_length=2048)
            device = next(self.model.parameters()).device
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=250,
                    temperature=0.3,
                    top_p=0.85,
                    do_sample=True,
                    pad_token_id=self.tokenizer.pad_token_id if self.tokenizer.pad_token_id else self.tokenizer.eos_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                    repetition_penalty=1.2,
                )
            
            # CRITICAL: Only decode the NEW tokens, not the whole sequence!
            input_length = inputs['input_ids'].shape[1]
            generated_tokens = outputs[0][input_length:]
            answer = self.tokenizer.decode(generated_tokens, skip_special_tokens=True)
            
            # Clean up Qwen artifacts
            answer = answer.replace("<|im_end|>", "").strip()
            
            # Fallback if answer is too short or messy
            if len(answer) < 10:
                return self.extract_fallback_answer(knowledge_context)
            
            return answer
            
        except Exception as e:
            if verbose:
                print(f"LLM error: {e}")
            return self.extract_fallback_answer(knowledge_context)



        
    
    def extract_fallback_answer(self, context: str) -> str:
        """Extract answer when LLM fails"""
        if not context:
            return "I couldn't find relevant information."
        
        chunks = context.split('\n\n')
        best_chunk = chunks[0] if chunks else context
        
        if "Q:" in best_chunk and "A:" in best_chunk:
            answer = best_chunk.split("A:", 1)[1].strip()
            for stop in ["\n\nQ:", "\nQ:", "##"]:
                if stop in answer:
                    answer = answer.split(stop)[0].strip()
            return answer
        
        if ":" in best_chunk:
            parts = best_chunk.split(":", 1)
            if len(parts) > 1:
                answer = parts[1].strip()
                if "\n\n" in answer:
                    answer = answer.split("\n\n")[0].strip()
                return answer
        
        answer = best_chunk
        for stop in ["\n\nQ:", "\nQ:"]:
            if stop in answer:
                answer = answer.split(stop)[0].strip()
        
        return answer if len(answer) > 10 else "I couldn't extract a clear answer."
    
    def clean_answer(self, answer: str) -> str:
        """Clean up the generated answer"""
        if not answer:
            return answer
        
        answer = re.sub(r'(\w+)_(\w+):', lambda m: m.group(0).replace('_', ' ').title().replace(' :', ':'), answer)
        answer = re.sub(r'\.(?=[A-Z])', '. ', answer)
        answer = self.translate_german_terms(answer)
        
        return answer.strip()
    
    def translate_german_terms(self, text: str) -> str:
        """Translate German academic terms to English"""
        translations = {
            r'\bSWS\b': 'SWS (Semester Hours per Week)',
            r'\bECTS\b': 'ECTS (European Credit Transfer System)',
            r'\bWS\b': 'WS (Winter Semester)',
            r'\bSS\b': 'SS (Summer Semester)',
            r'\bCP\b': 'CP (Credit Points)',
            r'\bStA(\d+)\b': lambda m: f'StA{m.group(1)} (Study Paper - {m.group(1)} hours workload)',
            r'\bPräs(\d+)\b': lambda m: f'Präs{m.group(1)} (Presentation - {m.group(1)} minutes)',
        }
        
        for pattern, replacement in translations.items():
            if callable(replacement):
                text = re.sub(pattern, replacement, text)
            elif re.search(pattern, text) and not re.search(pattern + r'\s*\(', text):
                text = re.sub(pattern, replacement, text)
        
        return text

# ============================================================================
# CHUNK CREATION
# ============================================================================

def create_chunks(course_data, professor_data=None, 
                          university_data=None, aspo_data=None):

    all_chunks = []
    

    
   
    program_count = 0
    
    for program_name, details in course_data.items():
        
        # =======================
        # APPLICATION LINKS 
        # =======================
        if details.get('program_links', {}).get('course_details'):
            apply_link = details['program_links']['course_details']
            
            # MASSIVE repetition with varied phrasings
            link_chunks = [
                f"Q: How can I apply to {program_name}? A: You can apply here: {apply_link}",
                f"Q: How can I apply to the {program_name}? A: You can apply here: {apply_link}",
                f"Q: How do I apply to {program_name}? A: You can apply here: {apply_link}",
                f"Q: How do I apply for {program_name}? A: You can apply here: {apply_link}",
                f"Q: Where do I apply for {program_name}? A: You can apply here: {apply_link}",
                f"Q: Where can I apply for {program_name}? A: You can apply here: {apply_link}",
                f"Q: Application link for {program_name}? A: {apply_link}",
                f"Q: Apply to {program_name}? A: You can apply here: {apply_link}",
                f"Q: Application URL for {program_name}? A: {apply_link}",
                f"Q: {program_name} application link? A: {apply_link}",
                f"Q: Where to apply for {program_name}? A: {apply_link}",
                f"Application link for {program_name}: {apply_link}",
                f"To apply for {program_name} at Hof University, visit: {apply_link}",
                f"Apply for {program_name} here: {apply_link}",
                f"{program_name} application: {apply_link}",
            ]
            all_chunks.extend(link_chunks * 8)  # 8x repetition!
            program_count += len(link_chunks) * 8
        
        # =======================
        # CONTACTS 
        # =======================
        contacts = details.get('contacts', {})
        if contacts:
            contact_list = []
            for key, info in contacts.items():
                if isinstance(info, dict) and info.get('name'):
                    role = key.replace('_', ' ').title()
                    contact_list.append({
                        'role': role,
                        'name': info['name'],
                        'email': info['email']
                    })
            
            if contact_list:
                # Build comprehensive contact text
                if len(contact_list) == 1:
                    c = contact_list[0]
                    contact_text = f"{c['name']} ({c['role']}) - Email: {c['email']}"
                elif len(contact_list) == 2:
                    contact_text = f"{contact_list[0]['name']} ({contact_list[0]['role']}, {contact_list[0]['email']}) or {contact_list[1]['name']} ({contact_list[1]['role']}, {contact_list[1]['email']})"
                else:
                    contact_parts = [f"{c['name']} ({c['role']}, {c['email']})" for c in contact_list]
                    contact_text = ", ".join(contact_parts)
                
                # MASSIVE repetition for contact queries
                contact_chunks = [
                    f"Q: Who should I contact for {program_name}? A: You can reach out to {contact_text}",
                    f"Q: Who can I contact for {program_name}? A: You can reach out to {contact_text}",
                    f"Q: Contact person for {program_name}? A: You can reach out to {contact_text}",
                    f"Q: Contact persons for {program_name}? A: You can reach out to {contact_text}",
                    f"Q: Who should I reach out for {program_name}? A: You can reach out to {contact_text}",
                    f"Q: Who can I reach out for {program_name}? A: You can reach out to {contact_text}",
                    f"Q: Contact information for {program_name}? A: You can reach out to {contact_text}",
                    f"Q: Who to contact for {program_name}? A: You can reach out to {contact_text}",
                    f"Q: {program_name} contact? A: You can reach out to {contact_text}",
                    f"For {program_name}, contact: {contact_text}",
                    f"Contact information for {program_name}: {contact_text}",
                    f"{program_name} program contacts: {contact_text}",
                    f"Q: Wants to know more about {program_name}? A: You can reach out to {contact_text}",
                ]
                all_chunks.extend(contact_chunks * 8)  # 8x repetition!
                program_count += len(contact_chunks) * 8
        
        # =======================
        # ADMISSION REQUIREMENTS 
        # =======================
        admission_requirements = details.get('admission_requirements', {})
        if admission_requirements:
            # Check if there's actual content - try both possible key names
            has_academic = (admission_requirements.get('academic_requirements') or 
                           admission_requirements.get('academic') or '').strip()
            has_language = (admission_requirements.get('language_requirements') or 
                           admission_requirements.get('language') or '').strip()
            
            # Filter out invalid values
            if has_academic and has_academic.lower() in ['none', 'null', 'n/a', '']:
                has_academic = ''
            if has_language and has_language.lower() in ['none', 'null', 'n/a', '']:
                has_language = ''
            
            if has_academic or has_language:
                # Build the answer text
                answer_text = ""
                if has_academic:
                    answer_text += f"Academic requirements: {has_academic}. "
                if has_language:
                    answer_text += f"Language requirements: {has_language}. "
                
                req_chunks = [
                    f"Q: Admission requirements for {program_name}? A: {answer_text}",
                    f"Q: What are the admission requirements for {program_name}? A: {answer_text}",
                    f"Q: Admission requirements for {program_name.replace(' M.A.', '').replace(' B.A.', '').replace(' M.Sc.', '').replace(' B.Sc.', '')}? A: {answer_text}",
                    
                    f"Q: What are the entry requirements for {program_name}? A: {answer_text}",
                    f"Q: Entry requirements for {program_name}? A: {answer_text}",
                    f"Q: What do I need to apply for {program_name}? A: {answer_text}",
                    f"Q: Requirements for {program_name}? A: {answer_text}",
                    f"Q: What are the requirements for {program_name}? A: {answer_text}",
                    
                    f"Q: Am I eligible for {program_name}? A: {answer_text}",
                    f"Q: Eligibility criteria for {program_name}? A: {answer_text}",
                    f"Q: What qualifications do I need for {program_name}? A: {answer_text}",
                    
                    f"Q: Can you tell me the requirements for {program_name}? A: {answer_text}",
                    f"Q: Tell me about {program_name} admission requirements A: {answer_text}",
                    f"Q: What do I need to get into {program_name}? A: {answer_text}",
                    f"Q: How can I qualify for {program_name}? A: {answer_text}",
                    
                    f"Q: Admission requirements for {program_name.split(' M.')[0].split(' B.')[0]}? A: {answer_text}",
                    f"Q: What are the admission requirements for {program_name.split(' M.')[0].split(' B.')[0]}? A: {answer_text}",
                    f"Q: Requirements for {program_name.split(' M.')[0].split(' B.')[0]}? A: {answer_text}",
                ]
                
                all_chunks.extend(req_chunks * 10)
                
                # If has academic requirements
                if has_academic:
                    academic_chunks = [
                        f"Q: What are the academic requirements for {program_name}? A: {has_academic}",
                        f"Q: What academic qualifications do I need for {program_name}? A: {has_academic}",
                        f"Q: Tell me the academic criteria for {program_name} A: {has_academic}",
                        f"Q: What grades do I need for {program_name}? A: {has_academic}",
                    ]
                    req_chunks.extend(academic_chunks)
                
                # If has language requirements
                if has_language:
                    language_chunks = [
                        f"Q: What are the language requirements for {program_name}? A: {has_language}",
                        f"Q: Language requirements for {program_name}? A: {has_language}",
                        f"Q: Do I need English/German proficiency for {program_name}? A: {has_language}",
                        f"Q: What language tests are required for {program_name}? A: {has_language}",
                        f"Q: Tell me about language requirements for {program_name} A: {has_language}",
                        f"Q: Do I need IELTS or TOEFL for {program_name}? A: {has_language}",
                        f"Q: What language level is required for {program_name}? A: {has_language}",
                    ]
                    req_chunks.extend(language_chunks)


    
                # Add all chunks with repetition
                all_chunks.extend(req_chunks * 10)
                
            else:
                # No specific requirements available
                contact_hint = f"Please contact {contact_text} for assistance." if contact_text else "Please contact the program coordinator for assistance."
        
                no_req_chunks = [
                    f"Q: What are the admission requirements for {program_name}? A: No specific admission requirements information available for {program_name}. {contact_hint}",
                    f"Q: What do I need to apply for {program_name}? A: Specific admission requirements are not listed for {program_name}. {contact_hint}",
                    f"Q: Requirements for {program_name}? A: Detailed requirements are not specified for {program_name}. {contact_hint}",
                    f"Q: Can you tell me the entry requirements for {program_name}? A: No specific admission requirements information available for {program_name}. {contact_hint}",
                    f"Q: Admission requirements for {program_name}? A: Detailed requirements are not specified for {program_name}. {contact_hint}",
                ]
                all_chunks.extend(no_req_chunks * 5)

        
        
        # Detailed admission requirements chunks
        detailed_admission = details.get('detailed_admission_requirements', '')
        if detailed_admission and detailed_admission.strip():
            clean_admission = re.sub(r'\s+', ' ', detailed_admission).strip()
            if len(clean_admission) > 50:
                detailed_chunks = [
                    f"Q: Detailed admission requirements for {program_name}? A: {clean_admission}",
                    f"Q: What are the specific admission criteria for {program_name}? A: {clean_admission}",
                    f"{program_name} detailed admission requirements: {clean_admission}",
                ]
              #  all_chunks.extend(detailed_chunks * 5)
        
        master_thesis = details.get('master_thesis', '')
        if master_thesis and master_thesis.strip():
            clean_thesis = re.sub(r'\s+', ' ', master_thesis).strip()
            if len(clean_thesis) > 50:
                thesis_chunks = [
                    f"Q: Master thesis requirements for {program_name}? A: {clean_thesis}",
                    f"Q: What are the thesis requirements for {program_name}? A: {clean_thesis}",
                    f"{program_name} master thesis information: {clean_thesis}",
                ]
                all_chunks.extend(thesis_chunks * 10)
            else:
                # Short/generic thesis info - still add but with fewer repetitions
                thesis_chunks = [
                    f"Q: Master thesis requirements for {program_name}? A: {clean_thesis}",
                    f"{program_name} master thesis: {clean_thesis}",
                ]
                all_chunks.extend(thesis_chunks * 10)
        else:
            # No thesis information available
            no_info_chunks = [
                f"Q: Master thesis requirements for {program_name}? A: No specific master thesis information is currently available for {program_name}. Please contact the program coordinator or visit the program page for details.",
                f"Q: What are the thesis requirements for {program_name}? A: Specific thesis requirements for {program_name} are not listed. Please contact the program coordinator.",
                f"{program_name} master thesis information: Not available. Please contact the program coordinator for thesis requirements.",
            ]
            all_chunks.extend(no_info_chunks * 5)   
        

        
        # Language
        language = details.get('language_of_instruction', 'Not specified')
        lang_chunks = [
            f"Q: What language is {program_name} taught in? A: {language}",
            f"Q: Language of instruction for {program_name}? A: {language}",
            f"{program_name} is taught in {language}",
            f"Q: In which language is {program_name} offered? A: {language}",
            f"Q: What is the teaching language for {program_name}? A: {language}",
            f"Q: Is {program_name} taught in English or German? A: {language}",
            f"Q: Which language do professors use in {program_name}? A: {language}",
            f"Q: What language are the lectures for {program_name}? A: {language}",
            f"Q: Is the medium of instruction for {program_name} English? A: {language}",
            f"Q: Are classes for {program_name} conducted in {language}? A: The program is taught in {language}",
            f"Q: What medium is used to teach {program_name}? A: {language}",
        ]
        all_chunks.extend(lang_chunks * 5)
        program_count += len(lang_chunks) * 5
        
        # Tuition
        tuition = details.get('tuition_fees', 'Not specified')
        is_free = 'none' in tuition.lower() or 'free' in tuition.lower() or tuition == ''
        
        if is_free:
            tuition_chunks = [
                f"Q: Is there any tuition fee for {program_name}? A: No, {program_name} is tuition-free.",
                f"Q: Tuition fee for {program_name}? A: {program_name} has no tuition fees.",
                f"Q: Tuition for {program_name}? A: No tuition fees.",
            ]
        else:
            tuition_chunks = [
                f"Q: Tuition fee for {program_name}? A: {tuition}",
                f"Q: What is the tuition fee for {program_name}? A: {tuition}",
                f"Q: Tuition for {program_name}? A: {tuition}",
            ]
        all_chunks.extend(tuition_chunks * 10)
        program_count += len(tuition_chunks) * 10
        
        # Duration
        duration = details.get('duration', 'Not specified')
        duration_chunks = [
            f"Q: How many semester for {program_name}? A: {duration}",
            f"Q: Duration of {program_name}? A: {duration}",
            f"Q: How long is {program_name}? A: {duration}",
        ]
        all_chunks.extend(duration_chunks * 5)
        program_count += len(duration_chunks) * 5
        
        # Start semester
        start = details.get('start', 'Not specified')
        start_chunks = [
            f"Q: When does {program_name} start? A: {program_name} starts in {start}",
            f"Q: {program_name} starts in which semester? A: {start}",
            f"Q: Start semester for {program_name}? A: {start}",
        ]
        all_chunks.extend(start_chunks * 5)
        program_count += len(start_chunks) * 5
        
        # Application deadline
        if details.get('application_period'):
            deadline = details['application_period']
            deadline_chunks = [
                f"Q: What is the application deadline for {program_name}? A: Application period for {program_name}: {deadline}",
                f"Q: Application deadline for {program_name}? A: {deadline}",
                f"Q: Deadline for {program_name}? A: {deadline}",
                f"Q: Application periods for {program_name}? A: {deadline}",
                f"Q: Can I apply for this {program_name} in summer semester? A: Application period for {program_name}: {deadline}",
                f"Q: Can I apply for this {program_name} in winter semester? A: Application period for {program_name}: {deadline}",
                f"Q: When can I apply for {program_name}? A: {deadline}",
                f"Q: Application period for {program_name}? A: {deadline}",
                f"Q: When can I apply for {program_name}? A: {deadline}",
                f"Q: Application period for bachelor programs? A: Application periods vary by program. Please specify which program",
                f"Q: Application deadline for bachelor programs? A: Application deadlines vary by program. Please specify which program",
                f"Q: Application period for masters programs? A: Application periods vary by program. Please specify which program",
                f"Q: Application deadline for masters programs? A: Application deadlines vary by program. Please specify which program",
            ]
            all_chunks.extend(deadline_chunks * 8)
            program_count += len(deadline_chunks) * 8
        
        # Modules
        modules = details.get('modules', [])
        if modules:
            for module in modules:
                module_name = module.get('module_name', '')
                if not module_name:
                    continue
                
                credits = module.get('credits', 'N/A')
                exam_type = module.get('examination_type', 'N/A')
                
                module_chunks = [
                    f"Q: How many ECTS for {module_name}? A: {credits} ECTS (European Credit Transfer System)",
                    f"Q: ECTS for {module_name}? A: {credits} ECTS",
                    f"Q: Credits for {module_name}? A: {credits} ECTS (European Credit Transfer System) credits",
                    f"Q: How many credits is {module_name}? A: {credits} ECTS (European Credit Transfer System) credits",
                ]
                all_chunks.extend(module_chunks * 10)
                program_count += len(module_chunks) * 10
                
                exam_chunks = [
                    f"Q: Exam type for {module_name}? A: {exam_type}",
                    f"Q: What is the exam type for {module_name}? A: {exam_type}",
                    f"Q: Examination type for {module_name}? A: {exam_type}",
                    
                ]
                all_chunks.extend(exam_chunks * 10)
                program_count += len(exam_chunks) * 10
    
    
    
    # =======================
    # PROFESSOR CHUNKS
    # =======================
    
    if professor_data:
        prof_count = 0
        
        for prof in professor_data:
            name = prof['full_name']
            email = prof['contact']['email']
            phone = prof['contact'].get('phone_all', '')
            office = prof['office']
            teaching = prof.get('teaching_areas', 'Not specified')
            
            # Email chunks - MASSIVE repetition
            email_chunks = [
                f"Q: What is {name}'s email? A: {name}'s email is {email}",
                f"Q: Email for {name}? A: {email}",
                f"Q: {name} email? A: {email}",
                f"Q: Email address for {name}? A: {email}",
                f"{name} email: {email}",
                f"Contact {name} at {email}",
            ]
            all_chunks.extend(email_chunks * 10)
            prof_count += len(email_chunks) * 10
            
            # Phone chunks
            if phone:
                phone_chunks = [
                    f"Q: What is {name}'s phone number? A: {phone}",
                    f"Q: Phone for {name}? A: {phone}",
                    f"Q: {name} phone? A: {phone}",
                    f"{name} phone number: {phone}",
                ]
                all_chunks.extend(phone_chunks * 10)
                prof_count += len(phone_chunks) * 10
            
            # Office chunks
            office_chunks = [
                f"Q: Where is {name}'s office? A: {name}'s office is at {office}",
                f"Q: Office for {name}? A: {office}",
                f"Q: {name} office? A: {office}",
                f"{name} office location: {office}",
            ]
            all_chunks.extend(office_chunks * 10)
            prof_count += len(office_chunks) * 10
            
            # Teaching chunks
            if teaching != 'Not specified':
                teaching_chunks = [
                    f"Q: What does {name} teach? A: {name} teaches {teaching}",
                    f"Q: Teaching areas for {name}? A: {teaching}",
                    f"{name} teaches: {teaching}",
                    f"what courses {name} teaches: {teaching}",
                ]
                all_chunks.extend(teaching_chunks * 10)
                prof_count += len(teaching_chunks) * 10
        
    
    
    
    
    # =======================
    # FINANCING / COSTS CHUNKS (CRITICAL FIX)
    # =======================
    
    if university_data and 'financing' in university_data:
    
        finance_count = 0
        
        financing = university_data['financing']
        
        # Monthly expenses
        if 'expenses' in financing and 'monthly' in financing['expenses']:
            monthly = financing['expenses']['monthly']
            
            # Extract specific costs
            costs_map = {}
            for item in monthly:
                for key, value in item.items():
                    costs_map[key.lower()] = value
            
            # Accommodation/Rent
            if 'accommodation*' in costs_map:
                rent_info = costs_map['accommodation*']
                rent_amount = rent_info.split(',')[0] if ',' in rent_info else rent_info
                
                rent_chunks = [
                    f"Q: How much is rent in Hof? A: Average student accommodation cost: {rent_amount}",
                    f"Q: Cost of rent in Hof? A: {rent_amount}",
                    f"Q: Typical rent in Hof? A: {rent_amount}",
                    f"Q: Accommodation cost in Hof? A: {rent_amount}",
                    f"Q: Housing cost in Hof? A: {rent_amount}",
                    f"Q: How much for accommodation in Hof? A: {rent_amount}",
                    f"Q: Rent prices in Hof? A: {rent_amount}",
                    f"Student accommodation in Hof costs: {rent_amount}",
                    f"Typical rent for students in Hof: {rent_amount}",
                ]
                all_chunks.extend(rent_chunks * 10)  # 50x repetition!
                finance_count += len(rent_chunks) * 10
            
            # Health insurance
            if 'health insurance' in costs_map:
                insurance_info = costs_map['health insurance']
                insurance_amount = insurance_info.split(',')[0] if ',' in insurance_info else insurance_info
                
                insurance_chunks = [
                    f"Q: Cost of health insurance? A: Student health insurance costs: {insurance_amount}",
                    f"Q: How much is health insurance? A: {insurance_amount}",
                    f"Q: Health insurance price? A: {insurance_amount}",
                    f"Q: Monthly health insurance cost? A: {insurance_amount}",
                    f"Q: How much for health insurance? A: {insurance_amount}",
                    f"Student health insurance in Germany: {insurance_amount}",
                    f"Health insurance costs approximately: {insurance_amount}",
                ]
                all_chunks.extend(insurance_chunks * 10)
                finance_count += len(insurance_chunks) * 10
            
            # Food
            if 'food*' in costs_map:
                food_info = costs_map['food*']
                food_amount = food_info.split(',')[0] if ',' in food_info else food_info
                
                food_chunks = [
                    f"Q: How much is food cost per month? A: Average food expenses: {food_amount}",
                    f"Q: Monthly food cost? A: {food_amount}",
                    f"Q: Cost of food in Hof? A: {food_amount}",
                    f"Food expenses per month: {food_amount}",
                ]
                all_chunks.extend(food_chunks * 10)
                finance_count += len(food_chunks) * 10
            
            # Total monthly costs
            if 'total' in costs_map:
                total_info = costs_map['total']
                
                total_chunks = [
                    f"Q: What are the total costs per month? A: Total monthly expenses: {total_info}",
                    f"Q: Monthly living costs in Hof? A: {total_info}",
                    f"Q: How much do I need per month? A: Total monthly cost: {total_info}",
                    f"Q: Total monthly expenses? A: {total_info}",
                    f"Total monthly living costs for students: {total_info}",
                ]
                all_chunks.extend(total_chunks * 10)
                finance_count += len(total_chunks) * 10
        
        # Semester fees
        if 'expenses' in financing and 'per_semester' in financing['expenses']:
            semester_exp = financing['expenses']['per_semester']
            
            semester_costs = {}
            for item in semester_exp:
                for key, value in item.items():
                    semester_costs[key.lower()] = value
            
            if 'semester fee' in semester_costs:
                sem_fee_info = semester_costs['semester fee']
                sem_fee_amount = sem_fee_info.split(',')[0] if ',' in sem_fee_info else sem_fee_info
                
                sem_fee_chunks = [
                    f"Q: What is the semester fee? A: Semester fee: {sem_fee_amount}",
                    f"Q: How much is the semester fee? A: {sem_fee_amount}",
                    f"Q: Semester fee amount? A: {sem_fee_amount}",
                    f"Q: Cost of semester fee? A: {sem_fee_amount}",
                    f"Semester fee at Hof University: {sem_fee_amount}",
                ]
                all_chunks.extend(sem_fee_chunks * 10)
                finance_count += len(sem_fee_chunks) * 10
        
        # Working during studies
        if 'working_during_studies' in financing:
            work_info = financing['working_during_studies'].get('content', '')
            
            work_chunks = [
                f"Q: Can I work while studying? A: Yes, students can work with certain restrictions. EU students: up to 19 hours/week without social security contributions. Non-EU students: 140 full days or 280 half days per year.",
                f"Q: Am I allowed to work during studies? A: Yes. EU students can work up to 19 hours/week. Non-EU students are allowed 140 full days or 280 half days per year.",
                f"Q: Can I get a job while studying? A: Yes, students can work. EU students: max 19 hours/week. Non-EU: 140 full days/year or 280 half days/year.",
            ]
            all_chunks.extend(work_chunks * 10)
            finance_count += len(work_chunks) * 10
        
        # Scholarships
        if 'scholarships' in financing:
            scholarship_chunks = [
                f"Q: Are there scholarships available? A: Yes, Hof University offers scholarships including the Deutschlandstipendium (€300/month) and scholarships for senior students funded by the Bavarian Ministry and DAAD.",
                f"Q: Can I get any scholarships based on my results? A: Yes, the Deutschlandstipendium provides €300 per month for at least one year, based on talent and academic performance. It's open to all nationalities.",
                f"Q: Are scholarships available? A: Yes, multiple scholarship options including Deutschlandstipendium (€300/month) and scholarships for senior students.",
            ]
            all_chunks.extend(scholarship_chunks * 10)
            finance_count += len(scholarship_chunks) * 10


            # Tuition FAQs
        tuition_faq = [
            "Q: is there any tuition fees for bachelor programs? A: There is no tuition fees for bachelor programs.",
            "Q: is there any tuition fees for masters programs? A: Tuition fees vary by program. Some master programs are free, others charge fees. Please specify which program.",
        ]
        all_chunks.extend(tuition_faq * 10)
        
        # Application period FAQs
        app_period_faq = [
            "Q: application period for summer semester? A: Application deadlines vary by program. Please specify which program.",
            "Q: application period for winter semester? A: Application deadlines vary by program. Please specify which program.",
            "Q: What programs start in summer semester? A: Application deadlines vary by program. Please specify which program.",
            "Q: What programs start in winter semester? A: Application deadlines vary by program. Please specify which program.",

             
        ]
        all_chunks.extend(app_period_faq * 10)
        
     
    
    # =======================
    # VPD AND APPLICATION CHUNKS
    # =======================
    
    # University data chunks
    if university_data:
        if 'faqs' in university_data:
            for question, answer in university_data['faqs'].items():
                clean_answer = re.sub(r'<[^>]+>', '', answer)
                clean_answer = ' '.join(clean_answer.split())
                all_chunks.extend([f"Q: {question} A: {clean_answer}"] * 15)
        
        if 'general_admission_related_info' in university_data:
            admission_info = university_data['general_admission_related_info']
            
            # 5 Steps to apply - process each step
            if '5 Steps to apply Hof University' in admission_info:
                steps = admission_info['5 Steps to apply Hof University']
                for step in steps:
                    step_title = step.get('step_title', '')
                    step_content = step.get('content', '')
                    clean_text = re.sub(r'<[^>]+>', ' ', step_content)
                    clean_text = ' '.join(clean_text.split())
                    sentences = [s.strip() for s in re.split(r'[.!?]+', clean_text) if len(s.strip()) > 20]
                    step_answer = ' '.join(sentences[:8])
                    step_chunks = [
                        f"Q: What is the step '{step_title}' in the application process? A: {step_answer}",
                        f"Q: Can you explain the '{step_title}' step? A: {step_answer}",
                        f"Q: Tell me about '{step_title}' A: {step_answer}",
                        f"Q: How do I {step_title.lower()}? A: {step_answer}",
                    ]
                    all_chunks.extend(step_chunks * 8)
            
            # VPD needed programs
            if 'VPD_needed_programs' in admission_info:
                programs_list = admission_info['VPD_needed_programs']
                programs_text = ', '.join(programs_list)
                vpd_chunks = [
                    f"Q: Which programs require VPD (preliminary review document)? A: The following programs require VPD: {programs_text}",
                    f"Q: Do I need VPD for my program? A: VPD is required for the following programs: {programs_text}",
                    f"Q: What is VPD and which programs need it? A: VPD (preliminary review document) is required for: {programs_text}",
                    f"Q: Does my master's program require VPD? A: The programs requiring VPD are: {programs_text}",
                    f"Q: List all programs that need preliminary review document A: {programs_text}",
                ]
                all_chunks.extend(vpd_chunks * 8)
            
            # Basic application requirements of bachelor's programs
            if 'basic application requirements of bachelor\'s programs' in admission_info:
                bachelor_req_text = admission_info['basic application requirements of bachelor\'s programs']
                clean_text = re.sub(r'<[^>]+>', ' ', bachelor_req_text)
                clean_text = ' '.join(clean_text.split())
                sentences = [s.strip() for s in re.split(r'[.!?]+', clean_text) if len(s.strip()) > 20]
                bachelor_req_answer = ' '.join(sentences[:8])
                bachelor_req_chunks = [
                    f"Q: What are the basic application requirements for bachelor's programs? A: {bachelor_req_answer}",
                    f"Q: Application requirements for bachelor's programs? A: {bachelor_req_answer}",
                    f"Q: What do I need to apply for a bachelor's degree? A: {bachelor_req_answer}",
                    f"Q: Can I apply for bachelor's program with my qualifications? A: {bachelor_req_answer}",
                    f"Q: What are the admission requirements for undergraduate programs? A: {bachelor_req_answer}",
                    f"Q: How can I check if I'm eligible for bachelor's admission? A: {bachelor_req_answer}",
                    f"Q: What documents do I need for bachelor's application? A: {bachelor_req_answer}",
                ]
                all_chunks.extend(bachelor_req_chunks * 8)
            
            # Basic application requirements of master's programs
            if 'basic application requirements of master\'s programs' in admission_info:
                master_req_text = admission_info['basic application requirements of master\'s programs']
                clean_text = re.sub(r'<[^>]+>', ' ', master_req_text)
                clean_text = ' '.join(clean_text.split())
                sentences = [s.strip() for s in re.split(r'[.!?]+', clean_text) if len(s.strip()) > 20]
                master_req_answer = ' '.join(sentences[:8])
                master_req_chunks = [
                    f"Q: What are the basic application requirements for master's programs? A: {master_req_answer}",
                    f"Q: Application requirements for master's programs? A: {master_req_answer}",
                    f"Q: Can I apply for a master's program? A: {master_req_answer}",
                    f"Q: What do I need to apply for master's degree? A: {master_req_answer}",
                    f"Q: Am I eligible for master's admission? A: {master_req_answer}",
                    f"Q: What are the entry requirements for postgraduate programs? A: {master_req_answer}",
                    f"Q: Do I need uni-assist for master's application? A: {master_req_answer}",
                ]
                all_chunks.extend(master_req_chunks * 8)
            
            # Language requirements for English taught programs
            if 'language requirements for english taught programs' in admission_info:
                english_lang_text = admission_info['language requirements for english taught programs']
                clean_text = re.sub(r'<[^>]+>', ' ', english_lang_text)
                clean_text = ' '.join(clean_text.split())
                sentences = [s.strip() for s in re.split(r'[.!?]+', clean_text) if len(s.strip()) > 20]
                english_lang_answer = ' '.join(sentences[:8])
                english_lang_chunks = [
                    f"Q: What are the language requirements for English taught programs? A: {english_lang_answer}",
                    f"Q: Do I need IELTS or TOEFL for English programs? A: {english_lang_answer}",
                    f"Q: What English test score do I need? A: {english_lang_answer}",
                    f"Q: How can I prove my English proficiency? A: {english_lang_answer}",
                    f"Q: What is the minimum TOEFL score required? A: {english_lang_answer}",
                    f"Q: Can I apply without English language certificate? A: {english_lang_answer}",
                ]
                all_chunks.extend(english_lang_chunks * 8)
            
            # Language requirements for German taught programs
            if 'language requirements for german taught programs' in admission_info:
                german_lang_text = admission_info['language requirements for german taught programs']
                clean_text = re.sub(r'<[^>]+>', ' ', german_lang_text)
                clean_text = ' '.join(clean_text.split())
                sentences = [s.strip() for s in re.split(r'[.!?]+', clean_text) if len(s.strip()) > 20]
                german_lang_answer = ' '.join(sentences[:8])
                german_lang_chunks = [
                    f"Q: What are the language requirements for German taught programs? A: {german_lang_answer}",
                    f"Q: Do I need to know German for admission? A: {german_lang_answer}",
                    f"Q: What German language level do I need? A: {german_lang_answer}",
                    f"Q: How can I prove my German language skills? A: {german_lang_answer}",
                    f"Q: Is B2 German certificate mandatory? A: {german_lang_answer}",
                    f"Q: Which German language test is accepted? A: {german_lang_answer}",
                ]
                all_chunks.extend(german_lang_chunks * 8)
            
            # Enrollment
            if 'Enrollment' in admission_info:
                enrollment_text = admission_info['Enrollment']
                clean_text = re.sub(r'<[^>]+>', ' ', enrollment_text)
                clean_text = ' '.join(clean_text.split())
                sentences = [s.strip() for s in re.split(r'[.!?]+', clean_text) if len(s.strip()) > 20]
                enrollment_answer = ' '.join(sentences[:8])
                enrollment_chunks = [
                    f"Q: How does the enrollment process work? A: {enrollment_answer}",
                    f"Q: How do I enroll after getting admission? A: {enrollment_answer}",
                    f"Q: What are the steps for enrollment? A: {enrollment_answer}",
                    f"Q: Can I enroll in person at the university? A: {enrollment_answer}",
                    f"Q: What do I need to complete enrollment? A: {enrollment_answer}",
                    f"Q: When can I enroll for my program? A: {enrollment_answer}",
                ]
                all_chunks.extend(enrollment_chunks * 15)
            
            # Campus Card
            if 'Campus Card' in admission_info:
                campus_card_text = admission_info['Campus Card']
                clean_text = re.sub(r'<[^>]+>', ' ', campus_card_text)
                clean_text = ' '.join(clean_text.split())
                sentences = [s.strip() for s in re.split(r'[.!?]+', clean_text) if len(s.strip()) > 20]
                campus_card_answer = ' '.join(sentences[:8])
                campus_card_chunks = [
                    f"Q: What is the Campus Card? A: {campus_card_answer}",
                    f"Q: How do I get my student ID? A: {campus_card_answer}",
                    f"Q: Where can I pick up my campus card? A: {campus_card_answer}",
                    f"Q: When will I receive my campus card? A: {campus_card_answer}",
                    f"Q: Can my campus card be sent to my home country? A: {campus_card_answer}",
                ]
                all_chunks.extend(campus_card_chunks * 8)


            if 'financing' in university_data:
                financing_data = university_data['financing']
            
                # Monthly Expenses
                if 'expenses' in financing_data and 'monthly' in financing_data['expenses']:
                    monthly_expenses = financing_data['expenses']['monthly']
                    
                    # Process each expense item
                    for expense_item in monthly_expenses:
                        for expense_name, expense_value in expense_item.items():
                            # Clean the text
                            clean_value = re.sub(r'<[^>]+>', ' ', str(expense_value))
                            clean_value = ' '.join(clean_value.split())
                            
                            if expense_name == 'Total':
                                total_chunks = [
                                    f"Q: How much money do I need per month in Germany? A: You should budget approximately {clean_value} per month for living expenses",
                                    f"Q: What is the monthly cost of living? A: Monthly living expenses are approximately {clean_value}",
                                    f"Q: What are the total monthly expenses? A: Total monthly expenses are around {clean_value}",
                                    f"Q: How much does it cost to live per month? A: Approximately {clean_value} per month",
                                    f"Q: What is my monthly budget as a student? A: You should plan for {clean_value} monthly",
                                ]
                                all_chunks.extend(total_chunks * 15)
                            elif expense_name == 'Accommodation*':
                                accommodation_chunks = [
                                    f"Q: How much is rent in Hof? A: {clean_value}",
                                    f"Q: What is the cost of accommodation? A: {clean_value}",
                                    f"Q: How much should I budget for housing? A: {clean_value}",
                                    f"Q: What does rent include? A: {clean_value}",
                                    f"Q: How expensive is student accommodation? A: {clean_value}",
                                ]
                                all_chunks.extend(accommodation_chunks * 15)
                            elif expense_name == 'Health insurance':
                                insurance_chunks = [
                                    f"Q: How much is health insurance? A: {clean_value}",
                                    f"Q: What is the cost of health insurance for students? A: {clean_value}",
                                    f"Q: Do I need health insurance? A: Yes, {clean_value}",
                                    f"Q: How much does student health insurance cost? A: {clean_value}",
                                ]
                                all_chunks.extend(insurance_chunks * 15)
                            elif expense_name == 'Food':
                                food_chunks = [
                                    f"Q: How much should I budget for food? A: {clean_value}",
                                    f"Q: What is the cost of food per month? A: {clean_value}",
                                    f"Q: How much does eating cost? A: {clean_value}",
                                    f"Q: What are the food expenses? A: {clean_value}",
                                ]
                                all_chunks.extend(food_chunks * 15)
                            elif expense_name == 'Personal Expenses':
                                personal_chunks = [
                                    f"Q: How much for personal expenses? A: {clean_value}",
                                    f"Q: What about other personal costs? A: {clean_value}",
                                    f"Q: How much should I budget for personal items? A: {clean_value}",
                                ]
                                all_chunks.extend(personal_chunks * 15)
                            elif expense_name == 'Cell phone contract':
                                phone_chunks = [
                                    f"Q: How much is a phone contract? A: {clean_value}",
                                    f"Q: What does a mobile plan cost? A: {clean_value}",
                                    f"Q: How much for cell phone? A: {clean_value}",
                                ]
                                all_chunks.extend(phone_chunks * 15)
                
                # Per Semester Expenses
                if 'expenses' in financing_data and 'per_semester' in financing_data['expenses']:
                    semester_expenses = financing_data['expenses']['per_semester']
                    
                    for expense_item in semester_expenses:
                        for expense_name, expense_value in expense_item.items():
                            clean_value = re.sub(r'<[^>]+>', ' ', str(expense_value))
                            clean_value = ' '.join(clean_value.split())
                            
                            if expense_name == 'Semester fee':
                                semester_fee_chunks = [
                                    f"Q: How much is the semester fee? A: {clean_value}",
                                    f"Q: What fees do I pay per semester? A: {clean_value}",
                                    f"Q: Do I need to pay semester fees? A: Yes, {clean_value}",
                                    f"Q: What is included in the semester fee? A: {clean_value}",
                                    f"Q: How much are university fees? A: {clean_value}",
                                ]
                                all_chunks.extend(semester_fee_chunks * 15)
                            elif expense_name == 'Tuition fee':
                                tuition_chunks = [
                                    f"Q: Do I have to pay tuition fees? A: {clean_value}",
                                    f"Q: How much is tuition? A: {clean_value}",
                                    f"Q: Are there tuition fees in Germany? A: {clean_value}",
                                    f"Q: Is studying free in Germany? A: {clean_value}",
                                    f"Q: What is the tuition cost? A: {clean_value}",
                                ]
                                all_chunks.extend(tuition_chunks * 15)
                            elif expense_name == 'Fee Structure':
                                grad_fee_chunks = [
                                    f"Q: Are there fees for Graduate School programs? A: {clean_value}",
                                    f"Q: Do Graduate School students pay tuition? A: {clean_value}",
                                    f"Q: What is the fee structure for Graduate School? A: {clean_value}",
                                ]
                                all_chunks.extend(grad_fee_chunks * 15)
                
                # One-time Expenses
                if 'expenses' in financing_data and 'one_time' in financing_data['expenses']:
                    onetime_expenses = financing_data['expenses']['one_time']
                    
                    for expense_item in onetime_expenses:
                        for expense_name, expense_value in expense_item.items():
                            clean_value = re.sub(r'<[^>]+>', ' ', str(expense_value))
                            clean_value = ' '.join(clean_value.split())
                            
                            if expense_name == 'Total':
                                onetime_total_chunks = [
                                    f"Q: What are the one-time costs when I arrive? A: One-time expenses are approximately {clean_value}",
                                    f"Q: How much money do I need initially? A: You should have {clean_value} for one-time expenses",
                                    f"Q: What initial costs should I expect? A: Initial one-time costs are around {clean_value}",
                                ]
                                all_chunks.extend(onetime_total_chunks * 15)
                            elif expense_name == 'Residence permit':
                                residence_chunks = [
                                    f"Q: How much does a residence permit cost? A: {clean_value}",
                                    f"Q: Do I need to pay for residence permit? A: {clean_value}",
                                    f"Q: What is the cost of residence permit? A: {clean_value}",
                                ]
                                all_chunks.extend(residence_chunks * 15)
                            elif expense_name == 'Rent deposit':
                                deposit_chunks = [
                                    f"Q: How much is the rent deposit? A: {clean_value}",
                                    f"Q: Do I need to pay a security deposit? A: Yes, {clean_value}",
                                    f"Q: What is the housing deposit? A: {clean_value}",
                                ]
                                all_chunks.extend(deposit_chunks * 15)
                
                # Working During Studies
                if 'working_during_studies' in financing_data:
                    working_content = financing_data['working_during_studies'].get('content', '')
                    clean_text = re.sub(r'<[^>]+>', ' ', working_content)
                    clean_text = ' '.join(clean_text.split())
                    sentences = [s.strip() for s in re.split(r'[.!?]+', clean_text) if len(s.strip()) > 20]
                    working_answer = ' '.join(sentences[:8])
                    
                    working_chunks = [
                        f"Q: Can I work while studying in Germany? A: {working_answer}",
                        f"Q: Am I allowed to work as an international student? A: {working_answer}",
                        f"Q: How many hours can I work as a student? A: {working_answer}",
                        f"Q: What are the work restrictions for students? A: {working_answer}",
                        f"Q: Can I work part-time during my studies? A: {working_answer}",
                        f"Q: Do I need a work permit to work in Germany? A: {working_answer}",
                        f"Q: How many days can I work per year? A: {working_answer}",
                        f"Q: What are the rules for student employment? A: {working_answer}",
                    ]
                    all_chunks.extend(working_chunks * 15)
            
          
                # Scholarships
                if 'scholarships' in financing_data:
                    scholarships = financing_data['scholarships']
                    
                    # Combine all scholarship information for general queries
                    all_scholarship_info = []
                    
                    if 'Scholarships for degree-seeking international students' in scholarships:
                        general_info = scholarships['Scholarships for degree-seeking international students']
                        clean_general = re.sub(r'<[^>]+>', ' ', general_info)
                        clean_general = ' '.join(clean_general.split())
                        all_scholarship_info.append(clean_general)
                    
                    if 'Germany Scholarship (Deutschlandstipendium)' in scholarships:
                        deutschland_info = scholarships['Germany Scholarship (Deutschlandstipendium)']
                        clean_deutschland = re.sub(r'<[^>]+>', ' ', deutschland_info)
                        clean_deutschland = ' '.join(clean_deutschland.split())
                        all_scholarship_info.append(clean_deutschland)
                    
                    if 'Scholarships for senior students' in scholarships:
                        senior_info = scholarships['Scholarships for senior students']
                        clean_senior = re.sub(r'<[^>]+>', ' ', senior_info)
                        clean_senior = ' '.join(clean_senior.split())
                        all_scholarship_info.append(clean_senior)
                    
                    # Combined comprehensive answer from actual data
                    comprehensive_scholarship_answer = ' '.join(all_scholarship_info)
                    
                    # General scholarship chunks ONLY - covers all scholarship questions
                    general_scholarship_chunks = [
                        f"Q: Are there scholarships available? A: {comprehensive_scholarship_answer}",
                        f"Q: Can I get a scholarship at Hof University? A: {comprehensive_scholarship_answer}",
                        f"Q: How can I get financial aid? A: {comprehensive_scholarship_answer}",
                        f"Q: Does Hof University offer scholarships? A: {comprehensive_scholarship_answer}",
                        f"Q: What scholarships can I apply for? A: {comprehensive_scholarship_answer}",
                        f"Q: Tell me about scholarships at Hof University. A: {comprehensive_scholarship_answer}",
                        f"Q: What financial support is available? A: {comprehensive_scholarship_answer}",
                        f"Q: How can I fund my studies? A: {comprehensive_scholarship_answer}",
                        f"Q: Can I get any scholarships based on my results? A: {comprehensive_scholarship_answer}",
                        f"Q: What scholarship opportunities exist? A: {comprehensive_scholarship_answer}",
                        f"Q: Are there any financial aid options? A: {comprehensive_scholarship_answer}",
                        f"Q: How do I apply for scholarships? A: {comprehensive_scholarship_answer}",
                        f"Q: What types of scholarships are offered? A: {comprehensive_scholarship_answer}",
                        f"Q: Can international students get scholarships? A: {comprehensive_scholarship_answer}",
                        f"Q: Are merit-based scholarships available? A: {comprehensive_scholarship_answer}",
                        f"Q: What is Deutschlandstipendium? A: {comprehensive_scholarship_answer}",
                        f"Q: How much is the Germany Scholarship? A: {comprehensive_scholarship_answer}",
                        f"Q: Can I apply for Deutschlandstipendium? A: {comprehensive_scholarship_answer}",
                        f"Q: What are the requirements for Germany Scholarship? A: {comprehensive_scholarship_answer}",
                        f"Q: Can senior students get scholarships? A: {comprehensive_scholarship_answer}",
                        f"Q: When can I apply for scholarships? A: {comprehensive_scholarship_answer}",
                        f"Q: Are there scholarships after enrollment? A: {comprehensive_scholarship_answer}",
                        f"Q: When is the scholarship application deadline? A: {comprehensive_scholarship_answer}",
                        f"Q: Can I apply for scholarships before enrollment? A: {comprehensive_scholarship_answer}",
                        f"Q: What scholarships can I apply for as a study applicant? A: {comprehensive_scholarship_answer}",
                        f"Q: Where can I find more scholarship information? A: {comprehensive_scholarship_answer}",
                        f"Q: What is DAAD scholarship database? A: {comprehensive_scholarship_answer}",
                    ]
                    all_chunks.extend(general_scholarship_chunks * 10)
            
    

        
        if 'academic_calendar' in university_data:
            all_holidays_vacations = []
            specific_holidays = {}  # Dictionary to store individual holidays
            
            for semester in university_data['academic_calendar']:
                for date_item in semester['dates']:
                    for event, date_value in date_item.items():
                        if any(kw in event for kw in ['Day', 'holiday', 'holidays', 'vacation', 
                                                       'Easter', 'Christmas', 'Pentecost', 
                                                       'Ascension', 'Corpus Christi', 'Labour', 
                                                       'Unity', 'Saints']):
                            all_holidays_vacations.append(f"{event}: {date_value}")
                            
                            # Store each specific holiday separately
                            holiday_name = event.lower()
                            specific_holidays[holiday_name] = {
                                'event': event,
                                'date': date_value
                            }
            
            all_holidays_text = ", ".join(all_holidays_vacations)
            
            # Create chunks for GENERAL queries (no semester mentioned)
            combined_holiday_chunks = [
                f"Q: Do we have class today? A: Please check if today falls on any of these dates when university is closed: {all_holidays_text}",
                f"Q: Is there class today? A: Classes may not be held on these dates: {all_holidays_text}",
                f"Q: Do I have class today? A: The university is closed on: {all_holidays_text}",
                f"Q: Does Hof University have class today? A: Check against these closure dates: {all_holidays_text}",
                f"Q: Are classes running today? A: Classes are not held on: {all_holidays_text}",
                f"Q: Will there be lectures today? A: No lectures on: {all_holidays_text}",
                f"Q: Do I need to go to university today? A: The university is closed on: {all_holidays_text}",
                f"Q: Should I come to campus today? A: Check if today is one of these closure dates: {all_holidays_text}",
                f"Q: Is the university open today? A: The university is closed on: {all_holidays_text}",
                f"Q: Is Hof University open today? A: University closure dates: {all_holidays_text}",
                f"Q: Is campus open today? A: Campus is closed on: {all_holidays_text}",
                f"Q: Is the school open today? A: The university is closed on: {all_holidays_text}",
                f"Q: Is today a holiday? A: University holidays and vacations: {all_holidays_text}",
                f"Q: Is it a holiday today? A: Check against these dates: {all_holidays_text}",
                f"Q: Are there classes today? A: Classes are not held on: {all_holidays_text}",
                f"Q: What are all the holidays? A: {all_holidays_text}",
                f"Q: Show me all university holidays A: {all_holidays_text}",
                f"Q: List all holidays A: {all_holidays_text}",
                f"Q: When is the university closed? A: {all_holidays_text}",
                f"Q: Show me all breaks and holidays A: {all_holidays_text}",
            ]
            all_chunks.extend(combined_holiday_chunks * 15)
            
            # Create chunks for EACH SPECIFIC HOLIDAY
            for holiday_key, holiday_info in specific_holidays.items():
                event_name = holiday_info['event']
                date_value = holiday_info['date']
                
                # Generate multiple question variations for each specific holiday
                specific_chunks = [
                    f"Q: When is {event_name}? A: {event_name} is on {date_value}",
                    f"Q: What date is {event_name}? A: {date_value}",
                    f"Q: {event_name} date? A: {date_value}",
                ]
                
                # Add variations based on holiday type
                if 'vacation' in holiday_key or 'holidays' in holiday_key:
                    specific_chunks.extend([
                        f"Q: When does {event_name} start? A: {event_name}: {date_value}",
                        f"Q: When is {event_name}? A: {date_value}",
                        f"Q: {event_name}? A: {date_value}",
                    ])
                
                if 'christmas' in holiday_key:
                    specific_chunks.extend([
                        f"Q: When is Christmas break? A: {date_value}",
                        f"Q: When is Christmas vacation? A: {date_value}",
                        f"Q: Christmas holiday dates? A: {date_value}",
                    ])
                
                if 'easter' in holiday_key:
                    specific_chunks.extend([
                        f"Q: When is Easter break? A: {date_value}",
                        f"Q: When are Easter holidays? A: {date_value}",
                        f"Q: Easter vacation dates? A: {date_value}",
                    ])
                
                if 'pentecost' in holiday_key:
                    specific_chunks.extend([
                        f"Q: When is Pentecost break? A: {date_value}",
                        f"Q: When is Whitsun? A: {date_value}",
                        f"Q: Pentecost holiday? A: {date_value}",
                    ])
                
                if 'labour' in holiday_key or 'labor' in holiday_key:
                    specific_chunks.extend([
                        f"Q: When is Labour Day? A: {date_value}",
                        f"Q: When is May Day? A: {date_value}",
                        f"Q: Is May 1st a holiday? A: Yes, {event_name}: {date_value}",
                    ])
                
                if 'ascension' in holiday_key:
                    specific_chunks.extend([
                        f"Q: When is Ascension Day? A: {date_value}",
                        f"Q: Ascension Day date? A: {date_value}",
                    ])
                
                if 'corpus christi' in holiday_key:
                    specific_chunks.extend([
                        f"Q: When is Corpus Christi? A: {date_value}",
                        f"Q: Corpus Christi date? A: {date_value}",
                    ])
                
                if 'unity' in holiday_key:
                    specific_chunks.extend([
                        f"Q: When is German Unity Day? A: {date_value}",
                        f"Q: When is Tag der Deutschen Einheit? A: {date_value}",
                        f"Q: Is October 3rd a holiday? A: Yes, {event_name}: {date_value}",
                    ])
                
                if 'saints' in holiday_key:
                    specific_chunks.extend([
                        f"Q: When is All Saints Day? A: {date_value}",
                        f"Q: When is Allerheiligen? A: {date_value}",
                        f"Q: Is November 1st a holiday? A: Yes, {event_name}: {date_value}",
                    ])
                
                if 'campus holiday' in holiday_key:
                    specific_chunks.extend([
                        f"Q: When is campus holiday? A: {date_value}",
                        f"Q: Campus closure date? A: {date_value}",
                    ])
                
                all_chunks.extend(specific_chunks * 10)  
        

    
            for semester in university_data['academic_calendar']:
                semester_name = semester['semester_name']
                semester_type = 'winter' if 'Winter' in semester_name else 'summer'

                next_semester_type = 'summer' if semester_type == 'winter' else 'winter'

                # Organize dates by category
                all_dates = {}
                semester_duration = ""
                lecture_start = ""
                lecture_end = ""
                exam_period = ""
                exam_registration = ""
                re_registration = ""
                holidays_and_vacations = []  # Combined list
                
                for date_item in semester['dates']:
                    for event, date_value in date_item.items():
                        all_dates[event] = date_value
                        
                        # Categorize events
                        if 'Duration of the semester' in event:
                            semester_duration = date_value
                        elif 'Start of the lecture period' in event or 'Start of lecture period' in event:
                            lecture_start = date_value
                        elif 'End of lecture period' in event:
                            lecture_end = date_value
                        elif 'Exam period' in event:
                            exam_period = date_value
                        elif 'Exam registration' in event:
                            exam_registration = date_value
                        elif 'Re-registration' in event:
                            re_registration = date_value
                        elif any(kw in event for kw in ['Day', 'holiday', 'holidays', 'vacation', 'Easter', 'Christmas', 'Pentecost', 'Ascension', 'Corpus Christi', 'Labour', 'Unity', 'Saints']):
                            # All holidays and vacations combined
                            holidays_and_vacations.append(f"{event}: {date_value}")
                
                # Semester Duration chunks
                if semester_duration:
                    duration_chunks = [
                        f"Q: When does {semester_type} semester start and end? A: {semester_type.capitalize()} semester {semester_name} runs from {semester_duration}",
                        f"Q: What is the duration of {semester_type} semester? A: {semester_duration}",
                        f"Q: When is {semester_type} semester {semester_name.split('/')[-1]}? A: It runs from {semester_duration}",
                        f"Q: What are the semester dates for {semester_name}? A: {semester_duration}",
                    ]
                    all_chunks.extend(duration_chunks * 10)
                
                # Lecture period chunks
                if lecture_start and lecture_end:
                    lecture_chunks = [
                        f"Q: When do lectures start in {semester_type} semester? A: Lectures start on {lecture_start}",
                        f"Q: When do classes begin for {semester_type} semester? A: Classes begin on {lecture_start}",
                        f"Q: When is the first day of classes in {semester_type} semester? A: {lecture_start}",
                        f"Q: When do lectures end in {semester_type} semester? A: Lectures end on {lecture_end}",
                        f"Q: What is the last day of classes in {semester_type} semester? A: {lecture_end}",
                        f"Q: Last class of {semester_type} semester? A: {lecture_end}",   
                        f"Q: What is the lecture period for {semester_type} semester? A: Lectures run from {lecture_start} to {lecture_end}",
                        f"Q: How long is the teaching period in {semester_type} semester? A: From {lecture_start} to {lecture_end}",
                    ]
                    all_chunks.extend(lecture_chunks * 10)
                
                # Exam period chunks
                if exam_period:
                    exam_chunks = [
                        f"Q: When are exams in {semester_type} semester? A: The exam period is {exam_period}",
                        f"Q: What is the exam period for {semester_type} semester? A: {exam_period}",
                        f"Q: When do exams start in {semester_type} semester? A: Exams start on {exam_period.split('-')[0].strip() if '-' in exam_period else exam_period.split('–')[0].strip()}",
                        f"Q: When is the examination period? A: For {semester_type} semester, exams are held during {exam_period}",
                        f"Q: How long is the exam period in {semester_type} semester? A: The exam period is {exam_period}",
                    ]
                    all_chunks.extend(exam_chunks * 10)
                
                # Exam registration chunks
                if exam_registration:
                    exam_reg_chunks = [
                        f"Q: When can I register for exams in {semester_type} semester? A: Exam registration is from {exam_registration}",
                        f"Q: What is the exam registration period? A: For {semester_type} semester, you can register from {exam_registration}",
                        f"Q: When does exam registration open? A: Exam registration opens from {exam_registration}",
                        f"Q: When is the deadline for exam registration? A: Exam registration closes on {exam_registration.split('-')[-1].strip() if '-' in exam_registration else exam_registration.split('–')[-1].strip()}",
                        f"Q: How do I register for exams? A: You can register during the exam registration period: {exam_registration}",
                    ]
                    all_chunks.extend(exam_reg_chunks * 10)
                
                # Re-registration chunks
                if re_registration:
                     re_reg_chunks = [
                        f"Q: When should I re-register for next semester? A: Re-registration period is {re_registration}",
                        f"Q: When should I re-register for {next_semester_type} semester? A: Re-registration is from {re_registration}",  # ← ADD THIS
                        f"Q: What is the re-registration deadline? A: You must re-register between {re_registration}",
                        f"Q: When can I enroll for the next semester? A: Re-registration opens from {re_registration}",
                        f"Q: When does re-registration start? A: Re-registration starts on {re_registration.split('-')[0].strip() if '-' in re_registration else re_registration.split('–')[0].strip()}",
                        f"Q: By when should I complete re-registration? A: Re-registration must be completed by {re_registration.split('-')[-1].strip() if '-' in re_registration else re_registration.split('–')[-1].strip()}",
                    ]
                    
                     all_chunks.extend(re_reg_chunks * 10)


                if holidays_and_vacations:
                    holidays_vacations_text = ", ".join(holidays_and_vacations)
                    holiday_vacation_chunks = [
                        # Semester-specific queries
                        f"Q: What are the holidays in {semester_type} semester? A: {holidays_vacations_text}",
                        f"Q: Which public holidays fall in {semester_type} semester? A: {holidays_vacations_text}",
                        f"Q: Are there any holidays during {semester_type} semester? A: {holidays_vacations_text}",
                        f"Q: When is the university closed for holidays in {semester_type} semester? A: {holidays_vacations_text}",
                        f"Q: List all holidays in {semester_name} A: {holidays_vacations_text}",
                        f"Q: What holidays do we have in {semester_type} semester? A: {holidays_vacations_text}",
                        f"Q: Holidays in {semester_type} semester? A: {holidays_vacations_text}",
                        f"Q: When are the vacations in {semester_type} semester? A: {holidays_vacations_text}",
                        f"Q: What is the semester break schedule for {semester_type} semester? A: {holidays_vacations_text}",
                        f"Q: When does {semester_type} semester vacation start? A: {holidays_vacations_text}",
                        f"Q: Are there any breaks during {semester_type} semester? A: {holidays_vacations_text}",
                        f"Q: What are the vacation periods in {semester_type} semester? A: {holidays_vacations_text}",
                        f"Q: Show me {semester_type} semester breaks and holidays A: {holidays_vacations_text}",
                    ]
                    all_chunks.extend(holiday_vacation_chunks * 10)
                
                
                
                # Welcome event chunks
                if 'Welcome event' in str(all_dates):
                    welcome_date = [v for k, v in all_dates.items() if 'Welcome event' in k]
                    if welcome_date:
                        welcome_chunks = [
                            f"Q: When is the welcome event for new students? A: The welcome event for first semester students is on {welcome_date[0]}",
                            f"Q: Is there an orientation for freshers? A: There's a welcome event on {welcome_date[0]}",
                            f"Q: When should I arrive as a new student? A: New students should attend the welcome event on {welcome_date[0]}",
                            f"Q: What is the orientation date? A: Orientation/welcome event is scheduled for {welcome_date[0]}",
                        ]
                        all_chunks.extend(welcome_chunks * 10)
                
                # Combined full calendar chunk
                all_events = [f"{event}: {date}" for event, date in all_dates.items()]
                full_calendar_text = "; ".join(all_events)
                full_calendar_chunks = [
                    f"Q: Show me the complete academic calendar for {semester_type} semester A: {full_calendar_text}",
                    f"Q: What are all the important dates for {semester_name}? A: {full_calendar_text}",
                    f"Q: Give me the full schedule for {semester_type} semester A: {full_calendar_text}",
                ]
                all_chunks.extend(full_calendar_chunks * 10)



    # ========================================================================
    # ASPO CHUNKS - FULL TEXT ALWAYS
    # ========================================================================
    if aspo_data:
        
        aspo_chunk_count = 0
        
        # Process HIGH PRIORITY sections
        if 'high_priority' in aspo_data:
            for section_name, full_text in aspo_data['high_priority'].items():
                
                
                if section_name == 'Repetition of Examinations':
                    qa_chunks = [
                        f"Q: How many times can I retake a failed exam? A: {full_text}",
                        f"Q: What happens if I fail an exam? A: {full_text}",
                        f"Q: When must I retake a failed exam? A: {full_text}",
                        f"Q: How many retakes are allowed? A: {full_text}",
                        f"Q: What happens if I don't take my retake exam within 6 months? A: {full_text}",
                        f"Q: Can I retake a passed exam to improve my grade? A: {full_text}",
                        f"Q: What is the deadline for first retake? A: {full_text}",
                        f"Q: What is the deadline for second retake? A: {full_text}",
                        f"Q: Repeat examination rules? A: {full_text}",
                        f"Repetition of Examinations: {full_text}",
                    ]
                    all_chunks.extend(qa_chunks * 15)
                    aspo_chunk_count += len(qa_chunks) * 15
                
                elif section_name == 'Registration for Examinations':
                    qa_chunks = [
                        f"Q: How do I register for exams? A: {full_text}",
                        f"Q: What happens if I don't register for an exam? A: {full_text}",
                        f"Q: Can I register late for exams? A: {full_text}",
                        f"Q: Exam registration process? A: {full_text}",
                        f"Q: When is the exam registration deadline? A: {full_text}",
                        f"Q: What if I miss exam registration? A: {full_text}",
                        f"Q: How to register for exams online? A: {full_text}",
                        f"Registration for Examinations: {full_text}",
                    ]
                    all_chunks.extend(qa_chunks * 15)
                    aspo_chunk_count += len(qa_chunks) * 15
                
                elif section_name == 'Grading  of Individual  Examinations':
                    qa_chunks = [
                        f"Q: What is the grading scale at Hof University? A: {full_text}",
                        f"Q: What is the minimum passing grade? A: {full_text}",
                        f"Q: How are grades calculated? A: {full_text}",
                        f"Q: What grade do I need to pass? A: {full_text}",
                        f"Q: What does grade 4.0 mean? A: {full_text}",
                        f"Q: Grading system at Hof University? A: {full_text}",
                        f"Q: What is a passing grade? A: {full_text}",
                        f"Grading of Individual Examinations: {full_text}",
                    ]
                    all_chunks.extend(qa_chunks * 15)
                    aspo_chunk_count += len(qa_chunks) * 15
                
                elif section_name == 'Standard Dates, Deadlines':
                    qa_chunks = [
                        f"Q: When do I need to complete first year exams? A: {full_text}",
                        f"Q: What is the deadline for exams? A: {full_text}",
                        f"Q: When must exams be taken? A: {full_text}",
                        f"Q: What happens if I miss the exam deadline? A: {full_text}",
                        f"Q: By when should I complete first semester exams? A: {full_text}",
                        f"Q: Exam deadlines for first year students? A: {full_text}",
                        f"Q: When must I take all my exams? A: {full_text}",
                        f"Standard Dates, Deadlines: {full_text}",
                    ]
                    all_chunks.extend(qa_chunks * 15)
                    aspo_chunk_count += len(qa_chunks) * 15
                
                elif section_name == 'Written Examinations':
                    qa_chunks = [
                        f"Q: What is a written examination? A: {full_text}",
                        f"Q: How are written exams conducted? A: {full_text}",
                        f"Q: Written exam rules? A: {full_text}",
                        f"Q: What are the rules during written exams? A: {full_text}",
                        f"Q: What should I bring to written exams? A: {full_text}",
                        f"Q: Written exam procedures? A: {full_text}",
                        f"Written Examinations: {full_text}",
                    ]
                    all_chunks.extend(qa_chunks * 15)
                    aspo_chunk_count += len(qa_chunks) * 15
                
                elif section_name == 'Oral Examinations':
                    qa_chunks = [
                        f"Q: What is an oral examination? A: {full_text}",
                        f"Q: How are oral exams conducted? A: {full_text}",
                        f"Q: Oral exam format? A: {full_text}",
                        f"Q: What happens during oral exams? A: {full_text}",
                        f"Q: Can oral exams be group exams? A: {full_text}",
                        f"Oral Examinations: {full_text}",
                    ]
                    all_chunks.extend(qa_chunks *15)
                    aspo_chunk_count += len(qa_chunks) * 15
                
                elif section_name == 'Presentations':
                    qa_chunks = [
                        f"Q: What is a presentation exam? A: {full_text}",
                        f"Q: How do presentation exams work? A: {full_text}",
                        f"Q: Presentation exam format? A: {full_text}",
                        f"Q: What is required in a presentation exam? A: {full_text}",
                        f"Q: How long should presentations be? A: {full_text}",
                        f"Presentations: {full_text}",
                    ]
                    all_chunks.extend(qa_chunks * 15)
                    aspo_chunk_count += len(qa_chunks) * 15
                
                elif section_name == 'Final Theses':
                    qa_chunks = [
                        f"Q: How do I submit my thesis? A: {full_text}",
                        f"Q: What are the thesis requirements? A: {full_text}",
                        f"Q: Thesis submission process? A: {full_text}",
                        f"Q: Can I change my thesis topic? A: {full_text}",
                        f"Q: What happens if I don't submit my thesis on time? A: {full_text}",
                        f"Q: How is the thesis graded? A: {full_text}",
                        f"Q: Who supervises my thesis? A: {full_text}",
                        f"Q: How do I choose a thesis supervisor? A: {full_text}",
                        f"Final Theses: {full_text}",
                    ]
                    all_chunks.extend(qa_chunks * 15)
                    aspo_chunk_count += len(qa_chunks) * 15
                
                elif section_name == 'Study Paper':
                    qa_chunks = [
                        f"Q: What is a study paper? A: {full_text}",
                        f"Q: Study paper requirements? A: {full_text}",
                        f"Q: How long do I have for a study paper? A: {full_text}",
                        f"Q: What happens if I don't submit study paper on time? A: {full_text}",
                        f"Study Paper: {full_text}",
                    ]
                    all_chunks.extend(qa_chunks * 15)
                    aspo_chunk_count += len(qa_chunks) * 15
                
                elif section_name == 'Take Home Exam':
                    qa_chunks = [
                        f"Q: What is a take home exam? A: {full_text}",
                        f"Q: How long for take home exam? A: {full_text}",
                        f"Q: Take home exam rules? A: {full_text}",
                        f"Q: What happens if I submit take home exam late? A: {full_text}",
                        f"Take Home Exam: {full_text}",
                    ]
                    all_chunks.extend(qa_chunks * 15)
                    aspo_chunk_count += len(qa_chunks) * 15
                
                elif section_name == 'Project Work':
                    qa_chunks = [
                        f"Q: What is project work? A: {full_text}",
                        f"Q: Project work requirements? A: {full_text}",
                        f"Q: How is project work evaluated? A: {full_text}",
                        f"Project Work: {full_text}",
                    ]
                    all_chunks.extend(qa_chunks * 15)
                    aspo_chunk_count += len(qa_chunks) * 15
                
                elif section_name == 'Portfolio Examination':
                    qa_chunks = [
                        f"Q: What is a portfolio examination? A: {full_text}",
                        f"Q: Portfolio exam format? A: {full_text}",
                        f"Q: How many parts in portfolio exam? A: {full_text}",
                        f"Portfolio Examination: {full_text}",
                    ]
                    all_chunks.extend(qa_chunks * 15)
                    aspo_chunk_count += len(qa_chunks) * 15
                
                elif section_name == 'Final Examination, Modules, Credit Points':
                    qa_chunks = [
                        f"Q: What does ECTS mean? A: {full_text}",
                        f"Q: How many hours is one credit point? A: {full_text}",
                        f"Q: What are credit points? A: {full_text}",
                        f"Q: How do I complete a module? A: {full_text}",
                        f"Q: What is a compulsory module? A: {full_text}",
                        f"Q: What is an elective module? A: {full_text}",
                        f"Q: How do I pass the final examination? A: {full_text}",
                        f"Final Examination, Modules, Credit Points: {full_text}",
                    ]
                    all_chunks.extend(qa_chunks * 15)
                    aspo_chunk_count += len(qa_chunks) * 15
        
        # Process MEDIUM PRIORITY sections
        if 'medium_priority' in aspo_data:
            for section_name, full_text in aspo_data['medium_priority'].items():
                
                # ✅ Always use full_text for all medium priority sections too
                
                if section_name == 'Extensions of Deadlines':
                    qa_chunks = [
                        f"Q: Can I get an extension for my exam deadline? A: {full_text}",
                        f"Q: How to request deadline extension? A: {full_text}",
                        f"Q: What are valid reasons for extension? A: {full_text}",
                        f"Q: Can I extend deadline due to illness? A: {full_text}",
                        f"Q: Extension for exam deadline? A: {full_text}",
                        f"Extensions of Deadlines: {full_text}",
                    ]
                    all_chunks.extend(qa_chunks * 10)
                    aspo_chunk_count += len(qa_chunks) * 10
                
                elif section_name == 'Compensation for Disadvantages':
                    qa_chunks = [
                        f"Q: Can I get accommodations for disabilities? A: {full_text}",
                        f"Q: How to request exam accommodations? A: {full_text}",
                        f"Q: What accommodations are available? A: {full_text}",
                        f"Q: Can I get extra time for exams? A: {full_text}",
                        f"Q: Disability accommodations for exams? A: {full_text}",
                        f"Compensation for Disadvantages: {full_text}",
                    ]
                    all_chunks.extend(qa_chunks * 10)
                    aspo_chunk_count += len(qa_chunks) * 10
                
                elif section_name == 'Dishonesty':
                    qa_chunks = [
                        f"Q: What happens if I cheat on an exam? A: {full_text}",
                        f"Q: Consequences of cheating? A: {full_text}",
                        f"Q: What is considered cheating? A: {full_text}",
                        f"Q: Plagiarism consequences? A: {full_text}",
                        f"Q: What if I bring unauthorized aids to exam? A: {full_text}",
                        f"Dishonesty: {full_text}",
                    ]
                    all_chunks.extend(qa_chunks * 10)
                    aspo_chunk_count += len(qa_chunks) * 10
                
                elif section_name == 'Violations  of Order':
                    qa_chunks = [
                        f"Q: What happens if I disrupt an exam? A: {full_text}",
                        f"Q: Exam violation consequences? A: {full_text}",
                        f"Q: Can I be excluded from an exam? A: {full_text}",
                        f"Violations of Order: {full_text}",
                    ]
                    all_chunks.extend(qa_chunks * 10)
                    aspo_chunk_count += len(qa_chunks) * 10
                
                elif section_name == 'General Admission Requirements':
                    qa_chunks = [
                        f"Q: What are the requirements to take an exam? A: {full_text}",
                        f"Q: Do I need to be matriculated to take exams? A: {full_text}",
                        f"Q: Exam admission requirements? A: {full_text}",
                        f"General Admission Requirements: {full_text}",
                    ]
                    all_chunks.extend(qa_chunks * 10)
                    aspo_chunk_count += len(qa_chunks) * 10
                
                elif section_name == 'Credit Transfer for Academic  Achievements and  Examination Results':
                    qa_chunks = [
                        f"Q: Can I transfer credits from another university? A: {full_text}",
                        f"Q: How do I transfer ECTS credits? A: {full_text}",
                        f"Q: Credit transfer process? A: {full_text}",
                        f"Credit Transfer for Academic Achievements and Examination Results: {full_text}",
                    ]
                    all_chunks.extend(qa_chunks * 10)
                    aspo_chunk_count += len(qa_chunks) * 10
                
                elif section_name == 'Pre-conditions  for Examination':
                    qa_chunks = [
                        f"Q: What are pre-conditions for examinations? A: {full_text}",
                        f"Q: Do I need attendance records for exams? A: {full_text}",
                        f"Q: Pre-requisites for taking exams? A: {full_text}",
                        f"Pre-conditions for Examination: {full_text}",
                    ]
                    all_chunks.extend(qa_chunks * 10)
                    aspo_chunk_count += len(qa_chunks) * 10
                
                elif section_name == 'Group Examinations':
                    qa_chunks = [
                        f"Q: How do group examinations work? A: {full_text}",
                        f"Q: Can exams be done in groups? A: {full_text}",
                        f"Q: Group exam requirements? A: {full_text}",
                        f"Group Examinations: {full_text}",
                    ]
                    all_chunks.extend(qa_chunks * 10)
                    aspo_chunk_count += len(qa_chunks) * 10
                
                elif section_name == 'Dual Studies':
                    qa_chunks = [
                        f"Q: What are dual studies? A: {full_text}",
                        f"Q: How do dual studies work? A: {full_text}",
                        f"Q: Requirements for dual studies? A: {full_text}",
                        f"Dual Studies: {full_text}",
                    ]
                    all_chunks.extend(qa_chunks * 10)
                    aspo_chunk_count += len(qa_chunks) * 10
                
                elif section_name == 'Retention  of Examination Documents':
                    qa_chunks = [
                        f"Q: How long are exam records kept? A: {full_text}",
                        f"Q: Exam document retention period? A: {full_text}",
                        f"Retention of Examination Documents: {full_text}",
                    ]
                    all_chunks.extend(qa_chunks * 10)
                    aspo_chunk_count += len(qa_chunks) * 10
                
                elif section_name == 'Examiners, Aids':
                    qa_chunks = [
                        f"Q: Who are my examiners? A: {full_text}",
                        f"Q: What aids are permitted in exams? A: {full_text}",
                        f"Q: How do I know permitted aids? A: {full_text}",
                        f"Examiners, Aids: {full_text}",
                    ]
                    all_chunks.extend(qa_chunks * 10)
                    aspo_chunk_count += len(qa_chunks) * 10

        
        

    
    return all_chunks


# ============================================================================
# INITIALIZATION FUNCTIONS
# ============================================================================

def load_course_data(file="data/all_programs_data.json"):
    with open(file, 'r', encoding='utf-8') as f:
        return json.load(f)

def load_professor_data(file="data/hof_university_staff.json"):
    try:
        with open(file, 'r', encoding='utf-8') as f:
            return json.load(f)
    except:
        return None

def load_university_data(file="data/general_hof_university_data.json"):
    try:
        with open(file, 'r', encoding='utf-8') as f:
            return json.load(f)
    except:
        return None

def load_aspo_data(file="data/aspo_priority_sections.json"):
    try:
        with open(file, 'r', encoding='utf-8') as f:
            return json.load(f)
    except:
        return None

def initialize_embedding_model(name='all-MiniLM-L6-v2'):
    print(f"Loading embedding model: {name}")
    return SentenceTransformer(name)

def create_faiss_index(chunks, embedding_model):
    embeddings = embedding_model.encode(chunks, show_progress_bar=True)
    d = embeddings.shape[1]
    index = faiss.IndexFlatL2(d)
    index.add(embeddings)
    
    return index

def initialize_llm_model(name="mistralai/Mistral-7B-v0.1", token=None):
    try:
        if token is None:
            token = os.environ.get("HF_TOKEN")
        
        
        tokenizer = AutoTokenizer.from_pretrained(name, token=token)
        
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Device: {device}")
        
        if device == "cuda":
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_use_double_quant=True
            )
            model = AutoModelForCausalLM.from_pretrained(
                name,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True,
                token=token
            )
        else:
            model = AutoModelForCausalLM.from_pretrained(
                name,
                trust_remote_code=True,
                token=token,
                torch_dtype=torch.float32
            )
        
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        print("LLM loaded")
        return model, tokenizer
        
    except Exception as e:
        print(f"✗ LLM loading failed: {e}")
        return None, None

def initialize_complete_system(load_llm=True, embedding_model='all-MiniLM-L6-v2', 
                              generation_model='Qwen/Qwen2.5-7B-Instruct', 
                              enable_reranking=True, reranker_top_k=5,
                              enable_memory=True):

    
    # 1. Load data
    print("\n[1/6] Loading data...")
    course_data = load_course_data()
    professor_data = load_professor_data()
    university_data = load_university_data()
    aspo_data = load_aspo_data()
    
    # 2. Load LLM FIRST — router needs it
    model, tokenizer = None, None
    if load_llm:
        print("\n[2/6] Loading LLM...")
        model, tokenizer = initialize_llm_model(generation_model, token=None)
    else:
        print("\n[2/6] Skipping LLM")

    # 3. Create router (replaces DynamicBackgroundMatcher)
    print("\n[3/6] Initializing LLMQueryRouter...")
    background_matcher = LLMQueryRouter(course_data, model, tokenizer)

    # 4. Create chunks
    print("\n[4/6] Creating chunks...")
    all_chunks = create_chunks(
        course_data,  professor_data, university_data, aspo_data
    )
    
    # 5. Embedding model
    print("\n[5/6] Initializing embedding model...")
    embedding_model_obj = initialize_embedding_model(embedding_model)
    
    # 6. FAISS index
    print("\n[6/6] Building FAISS index...")
    faiss_index = create_faiss_index(all_chunks, embedding_model_obj)
    
    rag_pipeline = RAGPipeline(
        model=model,
        tokenizer=tokenizer,
        embedding_model=embedding_model_obj,
        generation_model=generation_model,
        faiss_index=faiss_index,
        all_chunks=all_chunks,
        course_data=course_data,
        background_matcher=background_matcher,
        enable_reranking=enable_reranking,
        reranker_top_k=reranker_top_k,
        enable_memory=enable_memory
    )
    
    if enable_memory and model is not None:
        print(f"\nConversation Memory: ENABLED")
    elif enable_memory and model is None:
        print(f"\nConversation Memory: DISABLED (requires LLM)")
    else:
        print(f"\n  Conversation Memory: DISABLED (user choice)")
    
    if enable_reranking:
        print(f"Reranking: ENABLED (top_k={reranker_top_k})")
    else:
        print("  Reranking: DISABLED")
    

    
    return rag_pipeline

# ============================================================================
# TESTING AND DEMO
# ============================================================================

if __name__ == "__main__":
    # Initialize system with all features enabled
    rag_pipeline = initialize_complete_system(
        load_llm=True, 
        enable_reranking=True, 
        reranker_top_k=5,
        enable_memory=True  
    )
    

    test_conversations = [
    
        # Test set 1
        [
            "What master programs are available for computer science background?",
            "what are the addmission requirements for the 1st program",
            "What's the application deadline for it?",
            "is there any tuition fee for this program?",
            "Do I need health insurance to admit this program?",
            "what are the addmission requirements for the 2nd program",  # Topic change
            "What's the application deadline for it?",
            "tuition fee for this program?",
            "What is the teaching language for this program?",
            "What is the teaching language for Applied Research in Computer Science M.Sc?",  # Topic change
            "Is there any schoolarship available for master students?",  # Topic change
            "What is the rent amount in that area?",  # Topic change
        ],
    
        # Test set 2
        [
            "I'm looking for master programs are available for Management background without tuition fee",
            "My cgpa is 1.7 and IELTS score is 7. Can I apply for 1st program?",
            "What's the application deadline for it?",
            "is there any tuition fee for this program?",
            "What is the language requirements for this program?",
            "Can you give me more information about 3rd program?",  # Topic change
            "When can I apply for this program?",
            "Is there any tuition fee for this course?",
            "What is the grading scale at Hof University?",  # Topic change
        ],
    
        # Test set 3
        [
            "Is there any business related bachelor programs can I apply for?",
            "What are the requirements for 2nd program?",
            "What is the teaching language for this program?",
            "Is there any english instruct business bachelor programs available?",  # Topic change
            "What's the tuition fee for 1st program?",
            "When can I apply for this program?",
            "How long is this program?",
            "Admission requirements for this program?",
            "Can I work while studying?",  # Topic change
            "Are there any scholarships available for international students?",  # Topic change
        ],
    
        # Test set 4 - All questions are independent
        [
            "I got admission! How do I enroll?",
            "When is the enrollment period?",
            "What documents do I need for enrollment?",
            "Is health insurance mandatory?",
            "Can you recommend a health insurance provider?",
            "Do I need to pay anything during enrollment?",
            "When is the payment deadline and can it be extended?",
            "How do I get my student ID?",
            "When is course registration?",
        ],
    
        # Test set 5 - All questions are independent
        [
            "Can I repeat any exams?",
            "Is it possible to extand exam deadline?",
            "what happens when I failed one courses 3 times?",
            "Can I retake a passed exam to improve my grade?",
            "What happens if I cheat on an exam?",
        ],
    
        # Test set 6
        [
            "I studied computer science. Can I apply for AI and Robotics program?. What's the tuition fee for this program? Also when is the application period for this program?",
        ],
    
        # Test set 7
        [
            "How many ECTS credits for Operational Excellence & Innovation Management?",
            "What is the exam type for this course?",
            "what does mean by exam type 'StA10'?",
            "What is the exam type for International Value Chain Management?",  # Topic change
            "What are the thesis requirements for this course?",
        ],
    
        # Test set 8
        [
            "What master programs are available for Engineering background?",
            "what are the addmission requirements for the 1st program",
            "What's the application deadline for it?",
            "is there any tuition fee for this program?",
            "give me information about Operational Excellence?",  # Topic change
            "what are the requirements for this program?",
            "is there any tuition fee for this program?",
        ],
    
        # Test set 9
        [
            "I studied computer science. Can I apply for AI and Robotics program?",
            "Can I apply via an agency?",
            "What's the tuition fee for this program?",
            "Also when is the application period?",
            "How many semester for this program?",
            "Contact person for this program?",
            "Can I get any scholarships based on my results?",  # Topic change
        ],
    
        # Test set 10
        [
            "Class start date of summer semester",
            "Summer semester exam period?",
            "Holidays in winter semester?",  # Topic change
            "Do I have class today?",
            "Class start date of winter semester?",  # Topic change
            "Re-registration for the next summer semester",
        ],
    
        # Test set 11
        [
            "What is Professor Kirchner's email?",
            "What about his phone number?",
            "And his office location?",
            "What does he teach?",
            "Can I work while studying?",  # Topic change
        ],
    
        # Test set 12
        [
            "Are there tuition fees for bachelor programs?",
            "How much is the semester fee?",
            "What does the semester fee include?",
            "How much money do I need per month in Hof?",
            "What's the average rent for student accommodation?",
            "How much is health insurance?",
            "Can I work while studying?",  # Topic change
            "Are there any scholarships available?",  # Topic change
            "What is Deutschlandstipendium?",
            "How do I apply for scholarships?",
        ],
    
        # Test set 13
        [
            "I need to contact the Global Management program head. Who is it? Also, can i get program manager contact information? when can I apply for this program?",
        ],
    
        # Test set 14
        [
            "Is there any masters program for history background?",
            "Business",
            "what are the addmission requirements Applied Psychology?",
            "What's the application deadline for it?",
            "is there any tuition fee?",
            "How many semester for this program?",
            "Contact person for this program?",
        ],
    
        # Test set 15
        [
            "When is the next football worldcup?",  # Out of scope
            "What master programs are available for computer science background?",
        ],
    
    ]
    
    for conv_num, conversation in enumerate(test_conversations, 1):
        print(f"\n{'='*70}")
        print(f"CONVERSATION {conv_num}")
        print(f"{'='*70}")
        
        for turn_num, query in enumerate(conversation, 1):
            print(f"\n[Turn {turn_num}]")
            print(f"USER: {query}")
            print()
            
            answer = rag_pipeline.answer_query(query, verbose=True)
            
            print(f"\nASSISTANT: {answer}")
            print(f"\n{rag_pipeline.conversation_memory.get_conversation_summary()}")
            print("-" * 70)
        
        # Clear memory between conversations for demo
        if conv_num < len(test_conversations):
            print("\n[Clearing memory for next conversation demo]")
            rag_pipeline.conversation_memory = ConversationMemory(
                embedding_model=rag_pipeline.embedding_model,
                llm_model=rag_pipeline.model,
                llm_tokenizer=rag_pipeline.tokenizer
            )


[1/6] Loading data...

[2/6] Loading LLM...
Device: cuda


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

LLM loaded

[3/6] Initializing LLMQueryRouter...

[4/6] Creating chunks...

[5/6] Initializing embedding model...
Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


[6/6] Building FAISS index...


Batches:   0%|          | 0/2175 [00:00<?, ?it/s]

Initializing Reranker...
Loading cross-encoder model: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Cross-encoder loaded successfully!
Reranker initialized!
Conversation Memory initialized (Session-wise - ALL turns)!

Conversation Memory: ENABLED
Reranking: ENABLED (top_k=5)

CONVERSATION 1

[Turn 1]
USER: What master programs are available for computer science background?


ASSISTANT: 1. Applied Research in Computer Science M.Sc.
         2. Artificial Intelligence and Robotics M.Sc.
         3. Computer Science M.Sc.

Total turns: 1 | Last query: What master programs are available for computer sc...
----------------------------------------------------------------------

[Turn 2]
USER: what are the addmission requirements for the 1st program

[MEMORY] 1 previous turns
[RESOLVED] what are the addmission requirements for the 1st program → What are the admission requirements for the Applied Research in Computer Science M.Sc.?

ASSISTANT: Academic requirements for the Applied Research in Computer Science M. Sc.: Bachelor’s degree or similar in computer science, media informatics, mobile

## 2. Predefined conversation test cell

In [3]:
# ============================================================================
# FIXED TEST SET
# ============================================================================
test_questions = [
    "What master programs are available for computer science background?",
    "What are the admission requirements for 1st program?",
    "What is the tuition fee for this program?",
    "When is the next football world cup?",  # out-of-scope
]

rag_pipeline.conversation_memory.clear_session()

for i, question in enumerate(test_questions, 1):
    print(f"\n[Query {i}] {question}\n")
    answer = rag_pipeline.answer_query(question, verbose=False)
    if rag_pipeline._last_resolved_query != question:
        print(f"[Resolved] {rag_pipeline._last_resolved_query}\n")
    print(f"Answer: {answer}")
    print("-" * 50)

Session cleared

[Query 1] What master programs are available for computer science background?

Answer: 1. Applied Research in Computer Science M.Sc.
         2. Artificial Intelligence and Robotics M.Sc.
         3. Computer Science M.Sc.
--------------------------------------------------

[Query 2] What are the admission requirements for 1st program?

[Resolved] What are the admission requirements for the Applied Research in Computer Science M.Sc.?

Answer: Admission requirements for the Applied Research in Computer Science M. Sc. include:

- **Academic Requirements**: A bachelor's degree or similar in computer science, media informatics, mobile computing, or business information systems from an accredited university, with at least 210 ECTS (European Credit Transfer System) or equivalent; a minimum grade of 2.5 according to the German grading system (75%).
- **Accreditation**: Universities must be listed in Anabin with status H+.
- **Eligibility Restrictions**: Graduates from fields 

## 3. Interactive conversation test cell

In [4]:
rag_pipeline.conversation_memory.clear_session()

print("Chat started. Type 'new' to start a new conversation, 'quit' to exit.\n")

while True:
    query = input("You: ").strip()
    if not query:
        continue
    if query.lower() in ("quit", "exit"):
        break
    if query.lower() == "new":
        rag_pipeline.conversation_memory.clear_session()
        print("[New conversation started]\n")
        continue
    answer = rag_pipeline.answer_query(query, verbose=False)
    if rag_pipeline._last_resolved_query != query:
        print(f"\n[Resolved] {rag_pipeline._last_resolved_query} \n")
    print(f"Answer: {answer}\n")

Session cleared
Chat started. Type 'new' to start a new conversation, 'quit' to exit.



You:  What master programs are available for computer science background?


Answer: 1. Applied Research in Computer Science M.Sc.
         2. Artificial Intelligence and Robotics M.Sc.
         3. Computer Science M.Sc.



You:  What are the admission requirements for 1st program?



[Resolved] What are the admission requirements for the Applied Research in Computer Science M.Sc.? 

Answer: Admission requirements for the Applied Research in Computer Science M. Sc. include:

- **Academic Requirements**: A bachelor's degree or similar in computer science, media informatics, mobile computing, or business information systems, or a comparable study program from an accredited university, requiring at least 210 ECTS (European Credit Transfer System) or equivalent depending on the home country; minimum grade 2.5 according to the German grading system (75%).
- **Accreditation**: Only graduates from accredited universities listed in Anabin with status H+ are eligible.
- **Eligibility Restrictions**: Graduates from fields such as mechanical engineering, civil engineering, electronics & communications engineering, or mathematics/physics are not eligible.
- **Additional Credits Requirement**: Students with fewer than 210 credits (ECTS (European Credit Transfer System)) may sti

You:  quit


## 4. Evaluation across 3 embedding models and 1 generation model
  ### - Embedding models: MiniLM-L6-v2, BGE-base-en-v1.5, Multi-QA-MPNet-base
  ### - Generation model: Qwen2.5-7B-Instruct

In [3]:
# ============================================================================
# CELL 1: IMPORTS AND EVALUATOR CLASS
# ============================================================================

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
import json
from datetime import datetime
import time
import re
import os
import torch
from typing import Dict, List, Set, Tuple

class RAGEvaluator:
    def __init__(self, embedding_model_name='all-MiniLM-L6-v2'):
        """Initialize the enhanced RAG evaluator with embedding model"""
        print("Loading embedding model for evaluation...")
        self.embedding_model = SentenceTransformer(embedding_model_name)
        print("Embedding model loaded!")
    
    def extract_entities(self, text: str) -> Set[str]:
        """Extract numbers, dates, URLs, proper nouns (single + multi-word), and important words"""
        text_lower = text.lower()
        entities = set()
        
        # Numbers
        entities.update(re.findall(r'\b\d+\.?\d*\b', text))
        
        # Dates
        entities.update(re.findall(r'\b\d{1,2}[./]\d{1,2}[./]\d{2,4}\b', text))
        
        # URLs
        entities.update(re.findall(r'https?://[^\s]+', text))
        
        # Proper nouns (single + multi-word)
        entities.update(re.findall(r'\b[A-Z][a-z]+\b', text))  # single-word
        entities.update(re.findall(r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)+\b', text))  # multi-word
        
        # Important words (5+ characters, excluding common words)
        common_words = {'what', 'when', 'where', 'which', 'there', 'their', 'would', 
                       'could', 'should', 'about', 'these', 'those', 'that', 'this', 
                       'with', 'from', 'have', 'been', 'were', 'they', 'will', 'your',
                       'more', 'than', 'then', 'them', 'some', 'other', 'into', 'only',
                       'such', 'make', 'over', 'very', 'even', 'most', 'also', 'after',
                       'before', 'through', 'being', 'under', 'while'}
        words = re.findall(r'\b[a-z]{5,}\b', text_lower)
        entities.update([w for w in words if w not in common_words])
        
        return set(e.lower() for e in entities)

        
    
    def precision_score(self, expected: str, actual: str) -> float:
        expected_entities = self.extract_entities(expected)
        actual_entities = self.extract_entities(actual)
        
        if len(actual_entities) == 0:
            return 0.0
        
        true_positives = len(expected_entities.intersection(actual_entities))
        return float(true_positives / len(actual_entities))
    
    def recall_score(self, expected: str, actual: str) -> float:
        expected_entities = self.extract_entities(expected)
        actual_entities = self.extract_entities(actual)
        
        if len(expected_entities) == 0:
            return 1.0
        
        true_positives = len(expected_entities.intersection(actual_entities))
        return float(true_positives / len(expected_entities))
    
    def f1_score(self, expected: str, actual: str) -> float:
        precision = self.precision_score(expected, actual)
        recall = self.recall_score(expected, actual)
        
        if precision + recall == 0:
            return 0.0
        
        return float(2 * (precision * recall) / (precision + recall))

    def entity_f1(self, expected: str, actual: str) -> float:
        precision = self.precision_score(expected, actual)
        recall = self.recall_score(expected, actual)
        if precision + recall == 0:
            return 0.0
        return 2 * precision * recall / (precision + recall)

    def token_f1(self, expected: str, actual: str) -> float:
        exp_tokens = set(expected.lower().split())
        act_tokens = set(actual.lower().split())
        if len(exp_tokens) == 0 and len(act_tokens) == 0:
            return 1.0
        intersection = exp_tokens & act_tokens
        precision = len(intersection) / len(act_tokens) if len(act_tokens) > 0 else 0.0
        recall = len(intersection) / len(exp_tokens) if len(exp_tokens) > 0 else 0.0
        if precision + recall == 0:
            return 0.0
        return 2 * precision * recall / (precision + recall)
        
    def cosine_similarity_score(self, expected: str, actual: str) -> float:
        expected_emb = self.embedding_model.encode([expected])
        actual_emb = self.embedding_model.encode([actual])
        similarity = cosine_similarity(expected_emb, actual_emb)[0][0]
        return float(similarity)
    
    def semantic_similarity_score(self, expected: str, actual: str) -> float:
        expected_emb = self.embedding_model.encode(expected, convert_to_tensor=True)
        actual_emb = self.embedding_model.encode(actual, convert_to_tensor=True)
        similarity = util.pytorch_cos_sim(expected_emb, actual_emb).item()
        return float(similarity)
    
    def token_overlap_score(self, expected: str, actual: str) -> float:
        expected_tokens = set(expected.lower().split())
        actual_tokens = set(actual.lower().split())
        
        if len(expected_tokens) == 0 and len(actual_tokens) == 0:
            return 1.0
        
        intersection = expected_tokens.intersection(actual_tokens)
        union = expected_tokens.union(actual_tokens)
        
        return len(intersection) / len(union) if len(union) > 0 else 0.0
    
    def answer_relevance_score(self, question: str, answer: str) -> float:
        question_emb = self.embedding_model.encode(question, convert_to_tensor=True)
        answer_emb = self.embedding_model.encode(answer, convert_to_tensor=True)
        relevance = util.pytorch_cos_sim(question_emb, answer_emb).item()
        return float(relevance)
    
    def hallucination_score(self, expected: str, actual: str) -> float:
        expected_entities = self.extract_entities(expected)
        actual_entities = self.extract_entities(actual)
        
        if len(actual_entities) == 0:
            return 0.0
        
        false_positives = len(actual_entities - expected_entities)
        return float(false_positives / len(actual_entities))
    
    def faithfulness_score(self, expected: str, actual: str) -> float:
        semantic_sim = self.semantic_similarity_score(expected, actual)
        hallucination = self.hallucination_score(expected, actual)
        key_info_coverage = 1.0 - hallucination
        
        expected_length = len(expected.split())
        actual_length = len(actual.split())
        
        if expected_length == 0:
            length_ratio = 1.0
        else:
            length_ratio = actual_length / expected_length
        
        if length_ratio <= 1.5:
            length_penalty = 1.0
        elif length_ratio <= 2.0:
            length_penalty = 0.9
        elif length_ratio <= 3.0:
            length_penalty = 0.7
        else:
            length_penalty = 0.5
        
        faithfulness = (semantic_sim * 0.4 + key_info_coverage * 0.6) * length_penalty
        return float(faithfulness)
    
    def answer_completeness(self, expected: str, actual: str) -> float:
        expected_sentences = [s.strip() for s in expected.split('.') if s.strip()]
        actual_sentences = [s.strip() for s in actual.split('.') if s.strip()]
        
        if not expected_sentences:
            return 1.0
        if not actual_sentences:
            return 0.0
        
        covered = 0
        for exp_sent in expected_sentences:
            exp_emb = self.embedding_model.encode(exp_sent, convert_to_tensor=True)
            max_similarity = 0.0
            
            for act_sent in actual_sentences:
                act_emb = self.embedding_model.encode(act_sent, convert_to_tensor=True)
                sim = util.pytorch_cos_sim(exp_emb, act_emb).item()
                max_similarity = max(max_similarity, sim)
            
            if max_similarity > 0.7:
                covered += 1
        
        return covered / len(expected_sentences)
    
    def contains_key_information(self, expected: str, actual: str) -> float:
        expected_numbers = set(re.findall(r'\d+', expected))
        actual_numbers = set(re.findall(r'\d+', actual))
        
        expected_caps = set(re.findall(r'\b[A-Z][a-z]+\b', expected))
        actual_caps = set(re.findall(r'\b[A-Z][a-z]+\b', actual))
        
        expected_urls = set(re.findall(r'https?://[^\s]+', expected))
        actual_urls = set(re.findall(r'https?://[^\s]+', actual))
        
        scores = []
        
        if expected_numbers:
            number_score = len(expected_numbers.intersection(actual_numbers)) / len(expected_numbers)
            scores.append(number_score)
        
        if expected_caps:
            caps_score = len(expected_caps.intersection(actual_caps)) / len(expected_caps)
            scores.append(caps_score)
        
        if expected_urls:
            url_score = len(expected_urls.intersection(actual_urls)) / len(expected_urls)
            scores.append(url_score)
        
        return np.mean(scores) if scores else 0.5
    
    def exact_match_score(self, expected: str, actual: str) -> float:
        return 1.0 if expected.strip().lower() == actual.strip().lower() else 0.0
    
    def calculate_all_metrics(self, question: str, expected: str, actual: str, 
                             latency: float = None) -> Dict[str, float]:
        metrics = {
            'precision': self.precision_score(expected, actual),
            'recall': self.recall_score(expected, actual),
            'entity_f1': self.entity_f1(expected, actual),
            'token_f1': self.token_f1(expected, actual),
            'cosine_similarity': self.cosine_similarity_score(expected, actual),
            'semantic_similarity': self.semantic_similarity_score(expected, actual),
            'token_overlap': self.token_overlap_score(expected, actual),
            'answer_relevance': self.answer_relevance_score(question, actual),
            'hallucination_score': self.hallucination_score(expected, actual),
            'faithfulness': self.faithfulness_score(expected, actual),
            'answer_completeness': self.answer_completeness(expected, actual),
            'key_information_coverage': self.contains_key_information(expected, actual),
            'exact_match': self.exact_match_score(expected, actual),
        }
        
        if latency is not None:
            metrics['latency_seconds'] = latency
        
        return metrics

print("Evaluator class loaded!")

Evaluator class loaded!


In [4]:
# ============================================================================
# CELL 2: CONVERSATION-AWARE EVALUATION FUNCTION
# ============================================================================

def evaluate_conversations(rag_pipeline, test_csv_path: str, model_name: str,
                            output_dir: str = './evaluation_results'):

    os.makedirs(output_dir, exist_ok=True)

    print(f"\n{'='*80}")
    print(f"EVALUATING: {model_name.upper()}")
    print(f"{'='*80}\n")

    print(f"Loading test cases from {test_csv_path}...")
    test_df = pd.read_csv(test_csv_path)

    # Fill any missing values to avoid float NaN errors
    test_df['expected_resolved_question'] = (
        test_df['expected_resolved_question']
        .fillna(test_df['question'])
    )
    test_df['expected_answer'] = test_df['expected_answer'].fillna("")
    test_df['question']        = test_df['question'].fillna("")

    print(f"Loaded {len(test_df)} turns across "
          f"{test_df['conversation_number'].nunique()} conversations\n")

    evaluator = RAGEvaluator()
    results   = []

    conversations = test_df.groupby('conversation_number', sort=True)

    print("Running evaluation...")
    print("-" * 80)

    for conv_id, conv_df in conversations:
        print(f"\n[Conversation {conv_id}] ({len(conv_df)} turns)")

        if hasattr(rag_pipeline, 'conversation_memory') and \
                rag_pipeline.conversation_memory is not None:
            rag_pipeline.conversation_memory.clear_session()

        # Reset last resolved query tracker
        rag_pipeline._last_resolved_query = None

        conv_df = conv_df.reset_index(drop=True)

        for turn_idx, row in conv_df.iterrows():
            question          = str(row['question'])
            expected_resolved = str(row['expected_resolved_question'])
            expected_answer   = str(row['expected_answer'])
            turn_num          = turn_idx + 1

            print(f"  Turn {turn_num}: {question[:60]}...")

            # Get answer and measure latency
            start_time    = time.time()
            actual_answer = rag_pipeline.answer_query(question)
            latency       = time.time() - start_time

            # Guard: ensure actual_answer is always a string
            if not isinstance(actual_answer, str):
                actual_answer = str(actual_answer) if actual_answer is not None else ""

            # Get the pipeline's actual resolved query (what it used for retrieval)
            actual_resolved = getattr(rag_pipeline, '_last_resolved_query', None)
            if not isinstance(actual_resolved, str) or not actual_resolved:
                actual_resolved = question  # fallback: unresolved = original question

            # ── METRIC SET 1: Query Resolution Quality ──────────────────
            # How well did the pipeline resolve the vague question?
            resolution_metrics = evaluator.calculate_all_metrics(
                question=question,           # original question as context
                expected=expected_resolved,  # what the resolved query should be
                actual=actual_resolved,      # what the pipeline actually resolved to
                latency=None
            )

            # ── METRIC SET 2: Answer Quality ────────────────────────────
            # How good is the final answer?
            answer_metrics = evaluator.calculate_all_metrics(
                question=expected_resolved,  # use expected resolved Q for relevance
                expected=expected_answer,    # ground truth answer
                actual=actual_answer,        # pipeline's answer
                latency=latency
            )

            results.append({
                # Identifiers
                'model':               model_name,
                'conversation_number': conv_id,
                'turn_number':         turn_num,

                # Texts
                'question':            question,
                'expected_resolved':   expected_resolved,
                'actual_resolved':     actual_resolved,
                'expected_answer':     expected_answer,
                'actual_answer':       actual_answer,

                # Resolution metrics (prefix: res_)
                'res_Semantic Similarity': resolution_metrics['semantic_similarity'],
                'res_Token-F1':            resolution_metrics['token_f1'],
                'res_Entity-F1':           resolution_metrics['entity_f1'],
                'res_Token Overlap':       resolution_metrics['token_overlap'],

                # Answer metrics (prefix: ans_)
                'ans_Entity-F1':           answer_metrics['entity_f1'],
                'ans_Token-F1':            answer_metrics['token_f1'],
                'ans_Cosine Similarity':   answer_metrics['cosine_similarity'],
                'ans_Semantic Similarity': answer_metrics['semantic_similarity'],
                'ans_Answer Relevance':    answer_metrics['answer_relevance'],
                'ans_Hallucination':       answer_metrics['hallucination_score'],
                'ans_Faithfulness':        answer_metrics['faithfulness'],
                'ans_Completeness':        answer_metrics['answer_completeness'],
                'ans_Key Info':            answer_metrics['key_information_coverage'],
                'ans_Exact Match':         answer_metrics['exact_match'],
                'Latency (s)':             latency,
            })

    results_df = pd.DataFrame(results)

    # ── Overall scores (answer quality, same formula as before) ──────────
    gen_quality  = (results_df['ans_Entity-F1'].mean()
                    + results_df['ans_Token-F1'].mean()
                    + results_df['ans_Semantic Similarity'].mean()) / 3
    faithfulness = (results_df['ans_Faithfulness'].mean()
                    + (1 - results_df['ans_Hallucination'].mean())
                    + results_df['ans_Key Info'].mean()) / 3
    completeness = results_df['ans_Completeness'].mean()
    overall      = 0.4 * gen_quality + 0.35 * faithfulness + 0.25 * completeness

    # Resolution score (average semantic similarity of resolved queries)
    resolution_score = results_df['res_Semantic Similarity'].mean()

    # ── Per-conversation breakdown ────────────────────────────────────────
    conv_summary = results_df.groupby('conversation_number').agg(
        turns                  = ('turn_number',              'max'),
        # Resolution
        res_semantic_sim       = ('res_Semantic Similarity',  'mean'),
        res_token_f1           = ('res_Token-F1',             'mean'),
        res_entity_f1          = ('res_Entity-F1',            'mean'),
        # Answer
        ans_entity_f1          = ('ans_Entity-F1',            'mean'),
        ans_token_f1           = ('ans_Token-F1',             'mean'),
        ans_semantic_sim       = ('ans_Semantic Similarity',  'mean'),
        ans_faithfulness       = ('ans_Faithfulness',         'mean'),
        ans_hallucination      = ('ans_Hallucination',        'mean'),
        ans_completeness       = ('ans_Completeness',         'mean'),
        avg_latency            = ('Latency (s)',              'mean'),
    ).reset_index()

    # ── Print summary ─────────────────────────────────────────────────────
    print("\n" + "="*80)
    print("EVALUATION SUMMARY")
    print("="*80)
    print(f"Model:               {model_name}")
    print(f"Total Turns:         {len(results_df)}")
    print(f"Total Conversations: {results_df['conversation_number'].nunique()}")
    print()
    print("── Query Resolution ──────────────────────────────────────────")
    print(f"  Resolution Score (Semantic Sim): {resolution_score:.4f}")
    print(f"  Resolution Token-F1:             {results_df['res_Token-F1'].mean():.4f}")
    print(f"  Resolution Entity-F1:            {results_df['res_Entity-F1'].mean():.4f}")
    print()
    print("── Answer Quality ────────────────────────────────────────────")
    print(f"  Overall Score:       {overall:.4f}")
    print(f"  Generation Quality:  {gen_quality:.4f}")
    print(f"  Faithfulness:        {faithfulness:.4f}")
    print(f"  Completeness:        {completeness:.4f}")
    print(f"  Entity-F1:           {results_df['ans_Entity-F1'].mean():.4f}")
    print(f"  Token-F1:            {results_df['ans_Token-F1'].mean():.4f}")
    print(f"  Semantic Similarity: {results_df['ans_Semantic Similarity'].mean():.4f}")
    print(f"  Answer Relevance:    {results_df['ans_Answer Relevance'].mean():.4f}")
    print(f"  Faithfulness:        {results_df['ans_Faithfulness'].mean():.4f}")
    print(f"  Hallucination:       {results_df['ans_Hallucination'].mean():.4f}")
    print(f"  Completeness:        {results_df['ans_Completeness'].mean():.4f}")
    print(f"  Avg Latency:         {results_df['Latency (s)'].mean():.2f}s")
    print("="*80 + "\n")

    # ── Save results ──────────────────────────────────────────────────────
    timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
    turns_path = f"{output_dir}/{model_name}_turn_results.csv"
    conv_path  = f"{output_dir}/{model_name}_conv_summary.csv"

    results_df.to_csv(turns_path, index=False)
    conv_summary.to_csv(conv_path, index=False)
    print(f"Turn-level results:   {turns_path}")
    print(f"Conversation summary: {conv_path}\n")

    summary = {
        'model':                 model_name,
        'timestamp':             timestamp,
        'total_turns':           len(results_df),
        'total_conversations':   int(results_df['conversation_number'].nunique()),
        # Resolution
        'resolution_score':      float(resolution_score),
        'res_Token-F1':          float(results_df['res_Token-F1'].mean()),
        'res_Entity-F1':         float(results_df['res_Entity-F1'].mean()),
        # Answer
        'overall_score':         float(overall),
        'generation_quality':    float(gen_quality),
        'faithfulness_score':    float(faithfulness),
        'completeness_score':    float(completeness),
        'ans_Entity-F1':         float(results_df['ans_Entity-F1'].mean()),
        'ans_Token-F1':          float(results_df['ans_Token-F1'].mean()),
        'ans_Semantic Similarity': float(results_df['ans_Semantic Similarity'].mean()),
        'ans_Answer Relevance':  float(results_df['ans_Answer Relevance'].mean()),
        'ans_Faithfulness':      float(results_df['ans_Faithfulness'].mean()),
        'ans_Hallucination':     float(results_df['ans_Hallucination'].mean()),
        'ans_Completeness':      float(results_df['ans_Completeness'].mean()),
        'Latency (s)':           float(results_df['Latency (s)'].mean()),
    }

    return results_df, summary


print("Conversation evaluation function loaded!")


Conversation evaluation function loaded!


In [6]:
# ============================================================================
# CELL 3: RUN ALL EVALUATIONS
# ============================================================================

EMBEDDINGS = {
    'minilm':  'all-MiniLM-L6-v2',
    'bge':     'BAAI/bge-base-en-v1.5',
    'multiqa': 'sentence-transformers/multi-qa-mpnet-base-dot-v1',
}
GENERATIONS = {
    'qwen': 'Qwen/Qwen2.5-7B-Instruct',
}

TEST_CSV   = './15_testset_with_memory.csv'
OUTPUT_DIR = './evaluation_results/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

total_configs = len(EMBEDDINGS) * len(GENERATIONS)

print(f"Embedding Models:     {len(EMBEDDINGS)}")
print(f"Generation Models:    {len(GENERATIONS)}")
print(f"Total Configurations: {total_configs}")
print("="*80 + "\n")

all_summaries = []
config_num    = 0
start_time    = time.time()

for emb_name, emb_model in EMBEDDINGS.items():
    for gen_name, gen_model in GENERATIONS.items():
        config_num += 1
        model_id   = f"memory_{emb_name}_{gen_name}"

        print("\n" + "="*80)
        print(f"CONFIGURATION {config_num}/{total_configs}  —  {model_id}")
        print(f"  Embedding:  {emb_model}")
        print(f"  Generation: {gen_model}")
        print("="*80 + "\n")

        try:
            rag_pipeline = initialize_complete_system(
                embedding_model=emb_model,
                generation_model=gen_model,
                load_llm=True,
            )

            if not rag_pipeline:
                print(f"Failed to initialize {model_id}")
                continue

            results_df, summary = evaluate_conversations(
                rag_pipeline=rag_pipeline,
                test_csv_path=TEST_CSV,
                model_name=model_id,
                output_dir=OUTPUT_DIR,
            )

            summary['embedding_model']  = emb_name
            summary['embedding_name']   = emb_model
            summary['generation_model'] = gen_name
            summary['generation_name']  = gen_model
            all_summaries.append(summary)

            print(f"Completed: {model_id}")

        except Exception as e:
            import traceback
            print(f"Error with {model_id}: {e}")
            print(traceback.format_exc())
            continue

        finally:
            try:
                del rag_pipeline
            except NameError:
                pass
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

# Cross-model comparison table
if all_summaries:
    comparison_df = pd.DataFrame(all_summaries).sort_values(
        'overall_score', ascending=False
    )

    comparison_path = f"{OUTPUT_DIR}/all_models_comparison.csv"
    comparison_df.to_csv(comparison_path, index=False)

    print("\n" + "="*80)
    print("FINAL COMPARISON")
    print("="*80)
    cols = ['model', 'overall_score', 'generation_quality',
            'faithfulness_score', 'completeness_score',
            'Latency (s)', 'total_turns', 'total_conversations']
    print(comparison_df[cols].to_string(index=False))
    print(f"\nSaved: {comparison_path}")

total_mins = (time.time() - start_time) / 60
print(f"\nTotal time: {total_mins:.1f} minutes")
print("="*80)

Embedding Models:     3
Generation Models:    1
Total Configurations: 3


CONFIGURATION 1/3  —  memory_minilm_qwen
  Embedding:  all-MiniLM-L6-v2
  Generation: Qwen/Qwen2.5-7B-Instruct


[1/6] Loading data...

[2/6] Loading LLM...
Device: cuda


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

LLM loaded

[3/6] Initializing LLMQueryRouter...

[4/6] Creating chunks...

[5/6] Initializing embedding model...
Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


[6/6] Building FAISS index...


...

Initializing Reranker...
Loading cross-encoder model: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Cross-encoder loaded successfully!
Reranker initialized!
Conversation Memory initialized (Session-wise - ALL turns)!

Conversation Memory: ENABLED
Reranking: ENABLED (top_k=5)

EVALUATING: MEMORY_MULTIQA_QWEN

Loading test cases from ./15_testset_with_memory.csv...
Loaded 95 turns across 15 conversations

Loading embedding model for evaluation...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!
Running evaluation...
--------------------------------------------------------------------------------

[Conversation 1] (12 turns)
Session cleared
  Turn 1: What master programs are available for computer science back...
  Turn 2: what are the addmission requirements for the 1st program...
  Turn 3: What's the application deadline for it?...
  Turn 4: is there any tuition fee for this program?...
  Turn 5: Do I need health insurance to admit this program?...
  Turn 6: what are the addmission requirements for the 2nd program...
  Turn 7: What's the application deadline for it?...
  Turn 8: tuition fee for this program?...
  Turn 9: What is the teaching language for this program?...
  Turn 10: What is the teaching language for Applied Research in Comput...
  Turn 11: Is there any schoolarship available for master students?...
  Turn 12: What is the rent amount in that area?...

[Conversation 2] (9 turns)
Session cleared
  Turn 1: I'm looking for master programs a